# Topology and Shared Error Exposure in Model Lineages
| Experiment | Question | Design | Main outputs |
|---|---|---|---|
| Error transmission | Can fitted lineage loadings predict how a root perturbation reaches descendants? | Freeze the graph, perturb one root, calibrate on one direction, and evaluate on other directions. | `propagation.csv`, `superposition.csv`, `ridge_identity.csv` |
| Population growth | How do individual and aggregate errors change as a lineage grows? | Train roots, stars, chains, and graphs with multiple parents at paired seeds and budgets. | `growth.csv`, `growth_paired_contrasts.csv` |
| Teacher choice | When does choosing the best teacher for each student differ from choosing the best student population? | Vary teacher quality and error overlap, select assignments on validation data, and evaluate on test data. | `choice.csv`, `choice_margins.csv`, `balanced_choice.csv` |

All data are **synthetic regression data**. The learners are closed-form ridge regressors and trained nonlinear neural networks. No pretrained language models are loaded. The ridge learner provides an exact linear benchmark and the neural learner measures how well a fixed linear transmission approximation describes actual nonlinear training. All reported prediction errors use the original target scale unless explicitly marked as a standardized diagnostic.

### Running and resuming

1. Run cells from top to bottom. `MODE` defaults to `full`; set the environment variable `LINEAGE_MODE=smoke` before the setup cell for a small execution check. Smoke results are not the paper's full experiment.
2. A CPU is sufficient.
3. In Colab, `USE_DRIVE=True` mounts Drive and writes to `/content/drive/MyDrive/lineage_dag_full`. Outside Colab, `LINEAGE_OUTPUT` can specify a destination; the default is `./lineage_full`.
4. Keep one active writer per result directory. After an interruption, reconnect, mount the same Drive, and run from the setup cell. Completed models are reused; incomplete neural models resume from their latest checkpoint when the configuration, cache keys, and environment match.

### Full configuration

| Setting | Value |
|---|---|
| Independent seeds | 2701–2708 |
| Input dimension | 8 |
| Train / validation / test observations | 192 / 256 / 1024 |
| Propagation graph size | 8 nodes |
| Growth graph sizes | 4, 8, and 16 nodes |
| Neural architecture | One hidden layer with 16 tanh units and a linear scalar output |
| Neural optimizer | Full-batch Adam, learning rate 0.01 |
| Neural update budgets | 200 and 600 for the linear propagation task; 600 elsewhere |
| Distillation weight | 0.7 for propagation and choice; 0.4 and 0.7 for growth |
| Ridge penalty | 0.05 for propagation and choice; 0 and 0.05 for growth |
| Local error-direction scale | 0.18 |
| Root perturbation amplitude | 0.15 |
| Teacher-choice population | Four students choosing between two teachers |
| Teacher amplitude gaps | 0, 0.10, 0.20, 0.35, 0.50, 0.80 |
| Teacher-direction overlaps | 0, 0.5, 0.9 |
| Checkpoint interval | 50 neural updates |
| Bootstrap | 2000 seed-level resamples with a fixed bootstrap seed |

The `smoke` configuration reduces sample sizes, graph sizes, update budgets, and the choice grid. The executable `CFG` dictionary below is the authoritative configuration.

### Integrated supplementary experiments

Sections A-G reproduce the original notebook. Sections H-I append the optimizer
comparison and the mechanism supplement, respectively. The original data generator,
training objectives, gradients, teacher construction, selection rule, and evaluation
formulas are retained.

The default inline presentation focuses on the 600-update optimizer comparison,
regularized neural propagation, population growth, and the teacher-count extension.
The displayed conditions are specified in code, independently of the sign of the
results. Complete raw tables and diagnostic summaries are saved to disk. Set
`SHOW_ALL_TABLES=True` to display the complete supplementary summaries.

**Colab:** select a CPU runtime and run all cells in order. Previously completed
models are reused when their cache signatures match. Missing models are trained.
Keep one active notebook writing to each cache directory. This notebook is
self-contained; the two supplementary notebooks do not need to be uploaded.

**Paths:**
Optimizer caches remain under `lineage_optimizer_check`; mechanism caches remain
under `lineage_rebuttal_integrated`.
The final export combines the original and supplementary tables and figures.


In [ ]:
# Repository controls: no scientific configuration is changed here.
import os
from pathlib import Path
SHOW_ALL_TABLES = False
AUTO_DOWNLOAD = True
RUN_LONG_OPTIMIZER = True
RUN_AMPLITUDE_OPTIMIZER = True
# Preserve the recorded Colab paths. Local users can set LINEAGE_SOURCE and outputs.
RECORDED_SOURCE = Path(os.environ.get(
    'LINEAGE_SOURCE', '/content/drive/MyDrive/lineage_dag_full/b23c247b3a094046'))
RECORDED_ZIP = Path(os.environ.get('LINEAGE_SOURCE_ZIP', str(RECORDED_SOURCE)+'_full_results.zip'))


In [ ]:
import os
# Limit thread overhead for small matrices without modifying PyTorch or CUDA.
for k in ['OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS']:
    os.environ.setdefault(k,'1')
import sys, json, time, hashlib, itertools, platform, subprocess, importlib.util, zipfile
from pathlib import Path
for pkg in ['numpy','pandas','matplotlib','threadpoolctl']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable,'-m','pip','install',pkg])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
THREAD_LIMIT = threadpool_limits(limits=1)
try:
    from IPython.display import display
except ImportError:
    display=print
# Full is the recorded protocol. Set LINEAGE_MODE=smoke before this cell for a small check.
MODE=os.environ.get('LINEAGE_MODE','full')
USE_DRIVE=os.environ.get('LINEAGE_USE_DRIVE','1')=='1'
try:
    import google.colab
    IN_COLAB=True
except ImportError:
    IN_COLAB=False
ROOT=Path(os.environ.get('LINEAGE_OUTPUT','/content/lineage_full' if IN_COLAB else './lineage_full'))
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT=Path(os.environ.get('LINEAGE_OUTPUT','/content/drive/MyDrive/lineage_dag_full'))
# All scientific settings remain identical to the uploaded notebook.
CFG=dict(version='dag_training_full_v1',mode=MODE,seeds=list(range(2701,2709)),d=8,
         train=192,validation=256,test=1024,nodes=16,prop_nodes=8,
         beta=.7,betas=[.4,.7],ridge=.05,local_noise=.18,delta=.15,
         hidden=16,budgets=[200,600],growth_sizes=[4,8,16],learning_rate=.01,
         save_every=50,choice_slots=4,quality_gaps=[0.,.1,.2,.35,.5,.8],
         teacher_error=.3,rhos=[0.,.5,.9],bootstrap_draws=2000,bootstrap_seed=20260924)
if MODE=='smoke':
    CFG.update(seeds=[3701,3702],train=64,validation=64,test=128,nodes=6,prop_nodes=4,
               budgets=[20,40],save_every=10,growth_sizes=[3,6],quality_gaps=[0.,.2],rhos=[0.,.9],bootstrap_draws=100)
elif MODE!='full':raise ValueError('MODE must be full or smoke')
# Frozen identifier of the original computational version; documentation edits preserve cache compatibility.
CODE_SHA='1ac6b1a53dbbb84842cf1ff69735acec25783f612a8b1cc61a82696fc900d01d'
# Python/NumPy versions participate in the run key, so an environment change may select a new folder.
protocol=dict(config=CFG,code_sha=CODE_SHA,numpy=np.__version__,python=platform.python_version())
KEY=hashlib.sha256(json.dumps(protocol,sort_keys=True).encode()).hexdigest()[:16]
OUT=ROOT/KEY
# Prefer the recorded source when its protocol matches; preserve its exact bytes.
REUSE_RECORDED=False
if MODE=='full' and (RECORDED_SOURCE/'protocol.json').is_file():
    recorded_protocol=json.loads((RECORDED_SOURCE/'protocol.json').read_text())
    if recorded_protocol.get('config')!=CFG or recorded_protocol.get('code_sha')!=CODE_SHA:
        raise ValueError('The recorded source protocol does not match the original experiment.')
    OUT=RECORDED_SOURCE
    REUSE_RECORDED=True
OUT.mkdir(parents=True,exist_ok=True)
START=time.time();COUNTS=dict(trained=0,reused=0)
def atomic_text(path,text):
    'Write UTF-8 text through a sibling temporary file, then replace the destination.'
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.tmp');temp.write_text(text,encoding='utf-8');os.replace(temp,path)
def atomic_npz(path,**obj):
    'Save compressed NumPy arrays atomically without adding an unwanted suffix.'
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name(path.name+'.tmp')
    with open(temp,'wb') as f:np.savez_compressed(f,**obj)
    os.replace(temp,path)
def csv(name,rows):atomic_text(OUT/name,pd.DataFrame(rows).to_csv(index=False))
def log(msg):
    'Print case progress, elapsed minutes, and new-versus-reused model counts.'
    print(f'{msg} | elapsed {(time.time()-START)/60:.1f} minutes | newly trained {COUNTS["trained"]} reused {COUNTS["reused"]}',flush=True)
if not REUSE_RECORDED:atomic_text(OUT/'protocol.json',json.dumps(protocol,indent=2))
# Split boundaries are shared by every learner and experiment.
N=CFG['train'];VEND=N+CFG['validation'];TR=slice(0,N);VA=slice(N,VEND);TE=slice(VEND,None)
TOPO=['roots','star','chain','multi_parent'];LEARNERS=['ridge','mlp'];LONG=max(CFG['budgets'])
print('Mode',MODE,'results directory',OUT)
print('Test data are used only for evaluation. Teacher assignments and fitted loadings use validation data. All configured settings are reported.')


## A. Data generation and the connection to the theory

### A1. Inputs, targets, and local error sources

For each seed, draw independent raw input rows $r\sim\mathcal N(0,I_d)$. Estimate the uncentered second-moment matrix using training rows only, set $B=(R_{\mathrm{train}}^\top R_{\mathrm{train}}/n)^{-1/2}$, and transform every split as $X=RB$. Consequently,
$$
\frac{X_{\mathrm{train}}^\top X_{\mathrm{train}}}{n}=I_d.
$$
Validation and test raw inputs are separate draws. Both receive the training-fitted transformation. No validation or test observations are used to estimate $B$.

The linear target is $f_*(x)=x^\top\theta_*$, where a Gaussian draw is normalized to give $\|\theta_*\|_2=1$. For node $i$, draw a local direction $\eta_i\sim\mathcal N(0,\sigma_{\mathrm{loc}}^2 I_d/d)$ with $\sigma_{\mathrm{loc}}=0.18$. Its local training target is
$$
y_i(x)=f_*(x)+x^\top\eta_i.
$$
The same $\eta_i$ is held fixed across topology, budget, and perturbation comparisons within a seed. Separate deterministic random streams generate inputs, the true direction, node-specific directions, and two orthonormal intervention directions. Increasing the number of nodes therefore preserves the data and directions of earlier nodes.

These are input-dependent local errors rather than independently resampled label noise at every observation. The directions are drawn independently across nodes, but their realized error functions may be correlated over evaluation inputs. The covariance calculation retains those realized cross terms. Across seeds, both data and neural initialization change; a seed is the independent experimental unit.

### A2. Training follows the directed graph

The code indexes nodes from zero. `W[i, j]` is the teacher-mixture weight on node $j$ when training node $i$. Only $j<i$ is allowed. Roots fit their own local targets. A student fits
$$
t_i(x)=(1-\beta)y_i(x)+\beta\sum_{j<i}W_{ij}f_j(x),
\qquad W_{ij}\geq0,\qquad\sum_jW_{ij}=1.
$$
Teachers are trained first and frozen before their predictions enter a student's target. Here $W$ contains mixture weights; $A$ below contains direct error-transmission loadings.

| Topology | Direct teachers of node $i$ |
|---|---|
| `roots` | No teachers for any node |
| `star` | Node 0 for each $i>0$ |
| `chain` | Node $i-1$ for each $i>0$ |
| `multi_parent` | Nodes 0 and $i-1$, with equal weights; duplicate parents at $i=1$ are merged |

Each update budget defines a complete lineage. In particular, a 600-update student uses predictions from 600-update teachers. The 200- and 600-update experiments have matched initialization seeds but separate model-cache entries.

### A3. Exact ridge benchmark

The ridge learner solves
$$
\widehat\theta_i=\arg\min_\theta\left\{\frac1n\|X_{\mathrm{train}}\theta-t_i\|_2^2+\lambda\|\theta\|_2^2\right\}.
$$
Under the linear target and training whitening, define $\ell_i=1$ for a root and $\ell_i=1-\beta$ for a student. The direct loading and local error are
$$
A_{ij}=\frac{\beta}{1+\lambda}W_{ij},\qquad
u_i(x)=\frac{x^\top(\ell_i\eta_i-\lambda\theta_*)}{1+\lambda}.
$$
These give
$$
e=Ae+u,\qquad\Phi=(I-A)^{-1},\qquad
\Sigma=\Phi\Omega\Phi^\top,\qquad\Omega=\operatorname{Cov}(u).
$$
The inverse exists because $A$ is strictly lower triangular. The common shrinkage term $-\lambda\theta_*$ is part of $u_i$, so regularization can introduce an additional shared component. The growth experiment reports $\lambda=0$ and $\lambda=0.05$ separately. The numeric vector named `c` in the ridge checks implements $\ell_i$ above; it is not the paper's accumulated task-loading vector.

### A4. Nonlinear learner and nonlinear target

The neural learner is
$$
f_\vartheta(x)=v^\top\tanh(U^\top x+b)+b_0.
$$
Every parameter is trained with full-batch Adam on mean squared loss. The hidden weights have Gaussian scale $1/\sqrt d$, output weights have Gaussian scale $1/\sqrt h$, and biases start at zero. Adam uses moment coefficients 0.9 and 0.999, numerical constant $10^{-8}$, and learning rate 0.01. The `lam` argument affects ridge training only; neural training has no ridge penalty.

The nonlinear-target intervention experiment replaces the truth by
$$
f_*(x)=x^\top\theta_*+0.3\sin(x_1x_2).
$$
Learner type and target type are separate design choices. A ridge learner remains linear in its fitted coefficients even when the target function is nonlinear. A neural learner is trained nonlinearly even on the linear target task. The explicit baseline error identity above is checked for ridge with the linear target; transfer across interventions is measured for both target types.

**The next cell** constructs and saves the inputs, truth direction, node directions, intervention directions, and split indices for every seed. It also checks the whitening identity numerically.

In [ ]:
def make_data(seed):
    'Generate one paired seed bank. Return X [samples, features], truth coefficients, node directions, and two orthonormal probe directions.'
    # Separate random streams preserve earlier nodes, truth, and directions as graph size grows.
    d=CFG['d'];raw=np.random.default_rng(seed).normal(size=(N+CFG['validation']+CFG['test'],d))
    # Whiten with the training second moment only; use the same transform on validation and test rows.
    eig,Q=np.linalg.eigh(raw[TR].T@raw[TR]/N);B=(Q/np.sqrt(eig))@Q.T;X=raw@B
    theta=np.random.default_rng(seed+100000).normal(size=d);theta/=np.linalg.norm(theta)
    noise=np.array([np.random.default_rng(seed*1000+i+200000).normal(size=d)*CFG['local_noise']/np.sqrt(d) for i in range(CFG['nodes'])])
    # QR supplies orthonormal coefficient directions for both interventions and teacher design.
    dirs=np.linalg.qr(np.random.default_rng(seed+300000).normal(size=(d,2)))[0].T
    assert np.max(abs(X[TR].T@X[TR]/N-np.eye(d)))<1e-10
    return X,theta,noise,dirs
def truth_fn(X,theta,task):
    'Evaluate the linear target, optionally adding the nonlinear interaction term.'
    return X@theta+(0.3*np.sin(X[:,0]*X[:,1]) if task=='nonlinear' else 0.)
def parents(kind,m):
    'Return W [student, teacher] in topological order with unit row sums for students and zero rows for roots.'
    W=np.zeros((m,m))
    if kind=='star':W[1:,0]=1
    elif kind=='chain':W[np.arange(1,m),np.arange(m-1)]=1
    elif kind=='multi_parent':
        for i in range(1,m):
            js=sorted(set([0,i-1]));W[i,js]=1/len(js)
    assert not np.any(np.triu(W));return W
DATA={}
for seed in CFG['seeds']:
    data_file=OUT/'data'/f'{seed}.npz'
    if data_file.is_file():
        with np.load(data_file,allow_pickle=False) as z:
            DATA[seed]=tuple(z[k].copy() for k in ['X','theta','noise','dirs'])
            for k,v in [('train_indices',np.arange(N)),
                        ('validation_indices',np.arange(N,VEND)),
                        ('test_indices',np.arange(VEND,N+CFG['validation']+CFG['test']))]:
                np.testing.assert_array_equal(z[k],v)
        X,theta,noise,dirs=DATA[seed]
        assert X.shape==(N+CFG['validation']+CFG['test'],CFG['d'])
        np.testing.assert_allclose(X[TR].T@X[TR]/N,np.eye(CFG['d']),atol=1e-10)
    else:
        DATA[seed]=make_data(seed);X,theta,noise,dirs=DATA[seed]
        atomic_npz(data_file,X=X,theta=theta,noise=noise,dirs=dirs,
                   train_indices=np.arange(N),validation_indices=np.arange(N,VEND),
                   test_indices=np.arange(VEND,len(X)))
print('Data and split indices ready.')


## B. Learners, numerical checks, and checkpoint recovery

**The next cell** defines the two learners, the sequential graph-training routine, evaluation metrics, and the bootstrap helper. It first checks four representative neural gradient coordinates using centered finite differences.

### Saved state and reuse

Each completed model is saved in `model_cache/<signature>/completed.npz`. A model signature includes the full input array, training-target bytes, initialization seed, learner type, effective update budget, ridge-penalty argument, and the fixed computational-version identifier `CODE_SHA`. The enclosing run directory additionally depends on `CFG`, the NumPy version, and the Python version. Identical model requests reuse their completed predictions, including roots shared by several topologies.

Neural checkpoints contain the current parameters, Adam first and second moments, completed update count, and recorded training losses. They are written every 50 updates in full mode and at the last update. Writes use a temporary file followed by replacement. A restart restores this state and proceeds from the next update. Work since the last checkpoint may need to be repeated. Completed ridge models are saved after the closed-form solve.

The loader quarantines unreadable cache files before recreating the affected model. This handles damaged file reads, while signature assertions check the expected cache identity. The training loop stops if it encounters a nonfinite objective. These checks protect execution and do not establish the scientific hypotheses.

`CODE_SHA` is preserved from the original computational version for cache compatibility. It is a fixed identifier, not a freshly computed hash of this translated notebook. Future changes to numerical algorithms require a new computational identifier to avoid reusing stale results. Changing Python, NumPy, or configuration can also create a new run directory. A notebook's displayed outputs alone do not restore its variables or model cache.

### Evaluation quantities

For test errors $e_i(x)=f_i(x)-f_*(x)$, `metrics` computes
$$
\widehat L_{\mathrm{agg}}=\frac1{n_{\mathrm{test}}}\sum_x\left(\frac1M\sum_i e_i(x)\right)^2,
\qquad
\widehat L_{\mathrm{ind}}=\frac1M\sum_i\frac1{n_{\mathrm{test}}}\sum_x e_i(x)^2.
$$
It also records aggregate variance, squared aggregate bias, mean off-diagonal covariance, and effective rank $\operatorname{tr}(\widehat\Sigma)/\lambda_{\max}(\widehat\Sigma)$. Covariances use divisor $n_{\mathrm{test}}$ (`bias=True`) so that the finite-sample second-moment identities hold exactly on the evaluated sample. Effective rank is reported as missing when its denominator is essentially zero.

`bootstrap_mean` resamples the supplied seed-level values with replacement and returns their mean and the 2.5th and 97.5th percentiles of the resampled means. Paired comparisons supply one within-seed difference per seed. The neural loss histories also provide a final-interval decrease diagnostic; a fixed update budget does not itself establish convergence.

In [ ]:
def init_net(seed):
    'Initialize a one-hidden-layer tanh network reproducibly for one student slot.'
    r=np.random.default_rng(seed);d=CFG['d'];h=CFG['hidden']
    return [r.normal(size=(d,h))/np.sqrt(d),np.zeros(h),r.normal(size=h)/np.sqrt(h),np.zeros(1)]
def forward(X,p):
    'Return scalar predictions and hidden activations for all supplied rows.'
    H=np.tanh(X@p[0]+p[1]);return H@p[2]+p[3][0],H
def objective_grad(X,y,p):
    'Compute full-batch mean squared loss and analytic gradients for all four parameter arrays.'
    pred,H=forward(X,p);err=pred-y;n=len(y)
    g=2*err/n;dh=(g[:,None]*p[2])*(1-H*H)
    return float(np.mean(err**2)),[X.T@dh,dh.sum(0),H.T@g,np.array([g.sum()])]
# Check one representative coordinate in each parameter array before running the experiments.
pp=init_net(42);xx=np.random.default_rng(1).normal(size=(11,CFG['d']));yy=np.arange(11)/11
_,gg=objective_grad(xx,yy,pp)
for k,idx in [(0,(1,2)),(1,(2,)),(2,(3,)),(3,(0,))]:
    old=pp[k][idx];eps=1e-6
    pp[k][idx]=old+eps;lp=objective_grad(xx,yy,pp)[0]
    pp[k][idx]=old-eps;lm=objective_grad(xx,yy,pp)[0];pp[k][idx]=old
    assert abs((lp-lm)/(2*eps)-gg[k][idx])<1e-6
print('Neural-network finite-difference gradient check passed.')

MODEL_DIAGNOSTICS=[]
def safe_load(path):
    'Load a NumPy cache without pickle; quarantine unreadable files and signal that recomputation is needed.'
    try:
        with np.load(path,allow_pickle=False) as z:return {k:z[k].copy() for k in z.files}
    except (OSError,ValueError,EOFError,zipfile.BadZipFile) as exc:
        damaged=path.with_name(path.name+f'.corrupt_{time.time_ns()}');path.rename(damaged)
        print('Quarantined unreadable cache',path.name,type(exc).__name__);return None

def learner(kind,X,target,seed,budget,lam):
    'Fit or reuse a model and predict every split. Only X[TR] and target enter fitting; return predictions, training MSE, and cache signature.'
    # Ridge has a closed-form solution, so its cache ignores the displayed update-budget label.
    effective_budget=int(budget) if kind=='mlp' else 0
    # Hash the actual inputs and target so models can be reused across identical requests.
    sig=hashlib.sha256(X.tobytes()+target.tobytes()+str((seed,kind,effective_budget,lam,CODE_SHA)).encode()).hexdigest()
    folder=OUT/'model_cache'/sig;folder.mkdir(parents=True,exist_ok=True)
    done=folder/'completed.npz';saved=safe_load(done) if done.exists() else None
    if saved is not None:
        assert str(saved['signature'])==sig
        COUNTS['reused']+=1;return saved['pred'],float(saved['train_mse']),sig
    # Only the ridge branch uses lam as a penalty. The neural branch below is unregularized.
    if kind=='ridge':
        theta=np.linalg.solve(X[TR].T@X[TR]/N+lam*np.eye(X.shape[1]),X[TR].T@target/N)
        pred=X@theta;loss=float(np.mean((pred[TR]-target)**2))
        atomic_npz(done,signature=sig,pred=pred,train_mse=loss,theta=theta)
    else:
        p=init_net(seed);m=[np.zeros_like(a) for a in p];v=[np.zeros_like(a) for a in p];start=0;hist=[]
        ck=folder/'checkpoint.npz';state=safe_load(ck) if ck.exists() else None
        # Restore optimizer moments together with weights and the update counter for exact continuation.
        if state is not None:
            assert str(state['signature'])==sig
            start=int(state['step']);hist=state['history'].tolist()
            p=[state[f'p{k}'] for k in range(4)];m=[state[f'm{k}'] for k in range(4)];v=[state[f'v{k}'] for k in range(4)]
        for step in range(start+1,budget+1):
            loss,grad=objective_grad(X[TR],target,p)
            if not np.isfinite(loss):raise FloatingPointError(f'Nonfinite training loss {sig}')
            # Adam bias correction uses the absolute step, including steps completed before a restart.
            for k in range(4):
                m[k]=.9*m[k]+.1*grad[k];v[k]=.999*v[k]+.001*grad[k]**2
                p[k]-=CFG['learning_rate']*(m[k]/(1-.9**step))/(np.sqrt(v[k]/(1-.999**step))+1e-8)
            # Save post-update parameters and the matching post-update objective.
            if step%CFG['save_every']==0 or step==budget:
                post_loss=objective_grad(X[TR],target,p)[0];hist.append([step,post_loss])
                state=dict(signature=sig,step=step,history=np.array(hist))
                for k in range(4):state.update({f'p{k}':p[k],f'm{k}':m[k],f'v{k}':v[k]})
                atomic_npz(ck,**state)
        pred=forward(X,p)[0];loss=float(np.mean((pred[TR]-target)**2))
        atomic_npz(done,signature=sig,pred=pred,train_mse=loss,history=np.array(hist),**{f'p{k}':p[k] for k in range(4)})
    assert np.isfinite(pred).all()
    COUNTS['trained']+=1;return pred,loss,sig

VARIANTS={'base':(0,0),'h1':(1,0),'h2':(0,1),'sum':(1,1),'negative':(-1,0)}
def train_graph(seed,kind,topology,m,budget,beta,lam=.05,task='linear',variant='base'):
    'Train nodes in order with frozen parent predictions. Return predictions [nodes, samples], truth, mixture weights, and node training losses.'
    X,theta,noise,dirs=DATA[seed];truth=truth_fn(X,theta,task);W=parents(topology,m);pred=[];losses=[];ids=[]
    a,b=VARIANTS[variant]
    for i in range(m):
        local=truth[TR]+X[TR]@noise[i]
        if i==0:local=local+CFG['delta']*X[TR]@(a*dirs[0]+b*dirs[1])
        # A root fits its local target; each student mixes local targets with frozen parent predictions.
        target=local if not W[i].sum() else (1-beta)*local+beta*(W[i,:i]@np.array(pred))[:N]
        p,loss,sig=learner(kind,X,target,seed*100+i,budget,lam);pred.append(p);losses.append(loss);ids.append(sig)
    case=dict(seed=seed,learner=kind,topology=topology,M=m,budget=budget,beta=beta,lam=lam,task=task,variant=variant)
    casekey=hashlib.sha256(json.dumps(case,sort_keys=True).encode()).hexdigest()[:20]
    atomic_npz(OUT/'graph_predictions'/f'{casekey}.npz',pred=pred,truth=truth,W=W,train_mse=losses,
               case_json=json.dumps(case),model_ids=np.array(ids))
    return np.array(pred),truth,W,np.array(losses)

def metrics(pred,truth):
    'Evaluate a population on supplied data using raw error scale and empirical covariance with divisor n.'
    # Rows are models and columns are evaluation inputs; average over models before computing aggregate MSE.
    E=pred-truth;bar=E.mean(0);C=np.cov(E,bias=True);m=len(E);ev=np.linalg.eigvalsh(C)
    return dict(aggregate_mse=float(np.mean(bar**2)),variance=float(np.var(bar)),bias_squared=float(np.mean(bar)**2),
                individual_mse=float(np.mean(E**2)),offdiag_cov=float((C.sum()-np.trace(C))/(m*(m-1))),
                effective_rank=float(np.trace(C)/ev[-1]) if ev[-1]>1e-15 else np.nan)

def bootstrap_mean(values):
    'Return a mean and pointwise percentile interval by resampling independent seed-level values.'
    x=np.asarray(values,float);x=x[np.isfinite(x)]
    if len(x)<2:return (float(np.mean(x)) if len(x) else np.nan,np.nan,np.nan)
    rng=np.random.default_rng(CFG['bootstrap_seed']);b=x[rng.integers(0,len(x),(CFG['bootstrap_draws'],len(x)))].mean(1)
    return float(x.mean()),float(np.quantile(b,.025)),float(np.quantile(b,.975))


## C. Experiment 1 — propagation across a fixed lineage

This experiment asks whether the same fitted transmission loadings describe responses to interventions that were not used to fit them. It changes a root's local training target while holding the graph, all other local targets, initialization seeds, and update budgets fixed.

### Intervention arms and calibration

For each seed, learner, target task, budget, and topology, train five arms: `base`, `h1`, `h2`, `sum`, and `negative`. The root-target additions are respectively $0$, $\delta x^\top v_1$, $\delta x^\top v_2$, $\delta x^\top(v_1+v_2)$, and $-\delta x^\top v_1$, where $\delta=0.15$ and $v_1,v_2$ are orthonormal directions. Subtracting the baseline predictions gives the response $\Delta f_i$ at each node.

Using only validation responses to `h1`, fit a scalar incoming transmission coefficient for each student,
$$
\widehat a_i=
\frac{\left\langle\sum_jW_{ij}\Delta f_j,\Delta f_i\right\rangle}
{\left\|\sum_jW_{ij}\Delta f_j\right\|_2^2},
\qquad\widehat A_{ij}=\widehat a_iW_{ij}.
$$
For multiple parents, the known mixture weights are fixed and only their common multiplier is estimated. The fit is unconstrained and can produce a signed coefficient; it is a diagnostic of the proposed transmission approximation rather than a forced nonnegative fit. An effectively zero calibration denominator is marked as unidentified.

### Held-out prediction and metrics

For `h2` and `negative`, take the observed root response on test inputs as the starting signal and recursively predict descendant responses using $\widehat A$. Descendant test responses are used only for scoring. This evaluates propagation conditional on the root response rather than predicting the root's training response itself.

For each descendant and probe, record
$$
\mathrm{RMSE}=\sqrt{\operatorname{mean}(\widehat\Delta-\Delta)^2},\qquad
\mathrm{response\ RMS}=\sqrt{\operatorname{mean}\Delta^2},\qquad
\mathrm{relative\ RMSE}=\frac{\mathrm{RMSE}}{\mathrm{response\ RMS}},
$$
$$
\mathrm{skill}=1-\frac{\operatorname{mean}(\widehat\Delta-\Delta)^2}
{\operatorname{mean}\Delta^2}.
$$
A relative RMSE of zero is exact prediction, and a value of one matches the zero-response predictor's squared error. Skill is improvement over that predictor, not the usual centered explained variance. Ratios are missing when response energy is at most $10^{-16}$. Disconnected roots serve as a control with no descendant transmission and are excluded from the overall descendant relative-RMSE summary.

The `sum` arm tests superposition by comparing $\Delta f^{h_1+h_2}$ with $\Delta f^{h_1}+\Delta f^{h_2}$. The `negative` arm also tests reversal symmetry. These diagnostics use all nodes, including the root, whereas the propagation RMSE summary uses descendants. Linear-target neural runs use both 200 and 600 updates; nonlinear-target neural runs use 600. Ridge solves are closed form, so their recorded budget is a grouping label rather than a number of optimization steps.

For the linear-target ridge baseline, the code independently constructs $u$ and $\Phi$ and asserts that $e=\Phi u$ to within $10^{-9}$. Every completed topology writes updated CSV tables. The output files separate baseline-identity checks, held-out propagation accuracy, superposition, and training losses.

In [ ]:
PROP=[];SUPER=[];IDENT=[];TRAIN=[]
# Paired interventions change only the first root target. Other local directions and model seeds stay fixed.
for seed in CFG['seeds']:
    for task in ['linear','nonlinear']:
        for kind in LEARNERS:
            budgets=CFG['budgets'] if kind=='mlp' and task=='linear' else [LONG]
            for budget in budgets:
                for topo in TOPO:
                    banks={}
                    for variant in VARIANTS:
                        P,y,W,loss=train_graph(seed,kind,topo,CFG['prop_nodes'],budget,CFG['beta'],CFG['ridge'],task,variant)
                        banks[variant]=P
                        for i,l in enumerate(loss):TRAIN.append(dict(seed=seed,task=task,learner=kind,budget=budget,topology=topo,variant=variant,node=i,train_mse=l))
                    # Difference each intervention arm against its own matched baseline.
                    base=banks['base'];D={v:banks[v]-base for v in VARIANTS if v!='base'};m=len(W);A=np.zeros_like(W)
                    # Fit incoming loading multipliers using h1 validation responses only.
                    for i in range(1,m):
                        if not W[i].sum():continue
                        z=W[i]@D['h1'][:,VA];t=D['h1'][i,VA];den=z@z
                        A[i]=((z@t)/den)*W[i] if den>1e-16 else np.nan
                    # Hold the fitted A fixed across h2 and negative-direction test probes.
                    for probe in ['h2','negative']:
                        # Condition on the measured root response, then recursively predict descendants.
                        obs=D[probe][:,TE];hat=np.zeros_like(obs);hat[0]=obs[0]
                        for i in range(1,m):hat[i]=A[i,:i]@hat[:i]
                        for i in range(1,m):
                            err=np.mean((obs[i]-hat[i])**2);energy=np.mean(obs[i]**2)
                            PROP.append(dict(seed=seed,task=task,learner=kind,budget=budget,topology=topo,probe=probe,node=i,
                               fitted_loading=float(A[i].sum()),rmse=np.sqrt(err),response_rms=np.sqrt(energy),
                               relative_rmse=np.sqrt(err/energy) if energy>1e-16 else np.nan,
                               skill=1-err/energy if energy>1e-16 else np.nan))
                    # Superposition and reversal diagnostics use all nodes, including the intervened root.
                    for test,res,ref in [('superposition',D['sum']-D['h1']-D['h2'],D['h1']+D['h2']),('sign',D['negative']+D['h1'],D['h1'])]:
                        num=np.mean(res[:,TE]**2);den=np.mean(ref[:,TE]**2)
                        SUPER.append(dict(seed=seed,task=task,learner=kind,budget=budget,topology=topo,test=test,rmse=np.sqrt(num),relative_rmse=np.sqrt(num/den) if den>1e-16 else np.nan))
                    # Independent analytic baseline check for linear-target ridge training.
                    if kind=='ridge' and task=='linear':
                        X,theta,noise,dirs=DATA[seed];a=CFG['beta']/(1+CFG['ridge'])*W
                        c=np.where(W.sum(1)>0,1-CFG['beta'],1)
                        U=X[TE]@((c[:,None]*noise[:m]-CFG['ridge']*theta)/(1+CFG['ridge'])).T
                        Phi=np.linalg.solve(np.eye(m)-a,np.eye(m));error=base[:,TE]-y[TE]
                        gap=float(np.max(abs(error-Phi@U.T)));assert gap<1e-9
                        IDENT.append(dict(seed=seed,topology=topo,max_error=gap))
                    csv('propagation.csv',PROP);csv('superposition.csv',SUPER);csv('training.csv',TRAIN);csv('ridge_identity.csv',IDENT)
                    log(f'Propagation {seed} {task} {kind} {budget} {topo}')
print('Experiment 1 complete')


## D. Experiment 2 — population growth and aggregation

This experiment asks whether adding models improves the ensemble differently under different training graphs. Each graph is trained once at its largest size, then the first 4, 8, and 16 nodes form nested populations in full mode. An earlier node depends only on earlier teachers, so adding later nodes preserves its trained prediction. All evaluated nodes, including the root, receive weight $1/M$.

The target task is linear. Compare $\beta\in\{0.4,0.7\}$ using the longest neural budget. Ridge runs use both $\lambda=0$ and $\lambda=0.05$; neural runs use no ridge penalty. Matched seeds fix the input data, local directions, and initialization across topologies.

The main outcomes are raw aggregate MSE and average individual MSE, as defined in Section B. The table also includes aggregate bias and variance, average off-diagonal covariance, and effective rank. These are separately reported because changing the training graph can change individual quality as well as dependence.

### Two additional diagnostics

For a scale-controlled covariance diagnostic, estimate each model's error mean and standard deviation on validation data and apply that fixed transformation to its test errors. `standardized_aggregate_variance` is the variance of their average. This uses error labels for diagnosis and is not a new deployable predictor or a replacement for raw MSE.

For ridge models, compute $\Phi=(I-\beta W/(1+\lambda))^{-1}$ for each population prefix and the source-exposure vector
$$
v_j=\frac1M\sum_i\Phi_{ij},\qquad H=\sum_jv_j^2.
$$
Here `root_exposure` is $v_0$. Construct the realized local-error covariance $\widehat\Omega$ on test inputs and check that $v^\top\widehat\Omega v$ equals the observed aggregate variance. This retains correlated local errors and shrinkage effects. Finite-size curves illustrate diversification behavior; asymptotic claims are supplied by the theory.

**The next cell** saves `growth.csv` and matched comparisons with roots in `growth_paired_contrasts.csv`. A positive `aggregate_difference` means higher aggregate MSE than roots, while a negative `individual_difference` means lower average individual MSE. Their coexistence identifies the individual-versus-population tradeoff.

In [ ]:
GROW=[];PAIRS=[]
for seed in CFG['seeds']:
    for kind in LEARNERS:
        for beta in CFG['betas']:
            for lam in ([0.,CFG['ridge']] if kind=='ridge' else [0.]):
                for topo in TOPO:
                    P,y,W,loss=train_graph(seed,kind,topo,CFG['nodes'],LONG,beta,lam,'linear','base')
                    # Every prefix is a valid lineage; new nodes never change earlier predictions.
                    for size in CFG['growth_sizes']:
                        r=metrics(P[:size,TE],y[TE]);ev=P[:size,VA]-y[VA]
                        # Fit the diagnostic centering and scaling on validation errors only.
                        scale=ev.std(1);valid=bool(np.all(scale>1e-12))
                        standardized=(P[:size,TE]-y[TE]-ev.mean(1)[:,None])/np.maximum(scale[:,None],1e-12)
                        rec=dict(seed=seed,learner=kind,beta=beta,lam=lam,topology=topo,M=size,
                                 standardized_aggregate_variance=float(np.var(standardized.mean(0))) if valid else np.nan,**r)
                        if kind=='ridge':
                            Phi=np.linalg.solve(np.eye(size)-beta/(1+lam)*W[:size,:size],np.eye(size));expo=Phi.sum(0)/size
                            rec.update(root_exposure=expo[0],H=float(expo@expo))
                            X,theta,noise,dirs=DATA[seed];c=np.where(W[:size,:size].sum(1)>0,1-beta,1)
                            # Retain the full realized local covariance, including regularization-induced sharing.
                            U=X[TE]@((c[:,None]*noise[:size]-lam*theta)/(1+lam)).T
                            Omega=np.cov(U,rowvar=False,bias=True);predvar=expo@Omega@expo
                            assert abs(predvar-r['variance'])<1e-9
                            rec['variance_from_sources']=predvar
                        GROW.append(rec)
                    csv('growth.csv',GROW);log(f'Growth {seed} {kind} beta={beta} lambda={lam} {topo}')
gd=pd.DataFrame(GROW)
# Compare each topology to roots within the same seed, learner, budget setting, and population size.
for keys,g in gd.groupby(['seed','learner','beta','lam','M']):
    root=g[g.topology=='roots'].iloc[0]
    for _,r in g[g.topology!='roots'].iterrows():
        PAIRS.append(dict(zip(['seed','learner','beta','lam','M'],keys),topology=r.topology,
             aggregate_difference=r.aggregate_mse-root.aggregate_mse,individual_difference=r.individual_mse-root.individual_mse,
             aggregate_ratio=r.aggregate_mse/root.aggregate_mse))
csv('growth_paired_contrasts.csv',PAIRS)


## E. Experiment 3 — teacher quality, error overlap, and individual choice

This experiment creates two external teacher functions and actually trains students on their predictions. The teachers are specified analytically, allowing their quality gap and error directions to vary separately. For orthonormal directions $v_1,v_2$, define
$$
f_p(x)=f_*(x)+t\,x^\top v_1,\qquad
f_q(x)=f_*(x)+(t+\Delta)x^\top\left(\gamma v_1+\sqrt{1-\gamma^2}\,v_2\right),
\qquad t=0.3.
$$
Here $\Delta$ is an **error-amplitude gap**, not an MSE difference. The designed overlap $\gamma$ is the cosine between the coefficient directions. Whitening uses the training sample, so it need not equal the empirical error correlation on test inputs. The code records both teachers' test MSE and their measured test correlation. It retains the legacy name `rho` in variables, filenames, and CSV columns for this designed overlap; that field corresponds to $\gamma$ in the paper.

### Train a matched student bank

For each seed, learner, overlap, and amplitude gap, train four student slots under each teacher, producing eight student predictions. A slot uses the same local target, initialization seed, and update budget under both teachers. Students use $\beta=0.7$ and the longest budget. Ridge uses $\lambda=0.05$; the neural learner uses its unregularized squared-loss objective. The full grid has six gaps $\{0,0.1,0.2,0.35,0.5,0.8\}$ and three overlaps $\{0,0.5,0.9\}$ per learner.

### Choose on validation data, then evaluate on test data

1. **Individual rule.** Each student slot selects the teacher with the smaller individual validation MSE. Collect these four choices as `private`.
2. **Population rule.** Enumerate all $2^4=16$ assignments, compute the validation MSE of each four-student mean prediction, and choose the smallest. Save this assignment as `group`. Ties follow the fixed enumeration order.
3. Save both assignments to `frozen_choices` before computing their test scores. Define the test gain as
$$
\mathrm{gain}=\widehat L_{\mathrm{private}}-\widehat L_{\mathrm{group}}.
$$
Positive gain means that the validation-selected population rule improves test aggregate MSE. Population optimization is exhaustive over this finite menu on validation data, while its test advantage remains an empirical outcome.
4. For each slot, record its individual test margin, equal to the unchosen teacher's student MSE minus the chosen teacher's student MSE. A positive margin means the private preference selected on validation also holds on test data.

`joint_pattern` is true only when all four conditions hold: private choices concentrate on one teacher, the population assignment uses both teachers, every private preference has a positive individual test margin, and the population assignment has a positive test aggregate gain. `private_concentrated_count` counts only the first condition across seeds; `joint_pattern_count` counts all four together. Teacher concentration is $H=\sum_{p=1}^2\alpha_p^2$, so it is one under a common teacher and one half for a two-plus-two allocation.

### Balanced reuse as a separate comparison

The same bank supports an additional comparison that holds average individual performance fixed. Average the aggregate MSEs of the two concentrated assignments and compare this with the average aggregate MSE across all six two-plus-two assignments. Every slot–teacher student is used with weight one half in each collection, which is explicitly checked in the code. The difference therefore arises from how the predictions are grouped. This compares the **average of the two concentrated assignments**, not the best concentrated assignment, and is separate from the private-versus-population choice test.

The choice menu contains these two teachers at equal training budgets. It does not include autonomous training or teacher access costs. Every grid cell is reported. Seed-level bootstrap intervals are descriptive pointwise intervals for this exploratory grid.

In [ ]:
# rho is the legacy code/CSV name for the designed direction overlap gamma in the paper.
CHOICE=[];MARGINS=[];BALANCED=[];S=CFG['choice_slots'];configs=list(itertools.product(range(2),repeat=S))
for seed in CFG['seeds']:
    X,theta,noise,dirs=DATA[seed];truth=truth_fn(X,theta,'linear')
    for kind in LEARNERS:
        for rho in CFG['rhos']:
            for gap in CFG['quality_gaps']:
                # Prescribe teacher functions; train every slot under both teachers on matched local targets.
                teachers=np.array([truth+CFG['teacher_error']*(X@dirs[0]),truth+(CFG['teacher_error']+gap)*(X@(rho*dirs[0]+np.sqrt(1-rho*rho)*dirs[1]))])
                P=np.empty((S,2,len(X)))
                for s in range(S):
                    local=truth[TR]+X[TR]@noise[s]
                    for j in range(2):
                        target=(1-CFG['beta'])*local+CFG['beta']*teachers[j,TR]
                        P[s,j],_,_=learner(kind,X,target,seed*100+s,LONG,CFG['ridge'])
                # Individual choices and exhaustive population choices both use validation data.
                own=((P[:,:,VA]-truth[VA])**2).mean(2);private=own.argmin(1)
                val=[np.mean((P[np.arange(S),c][:,VA].mean(0)-truth[VA])**2) for c in configs]
                group=np.array(configs[int(np.argmin(val))])
                case=f'{seed}_{kind}_rho{rho}_gap{gap}'
                # Persist selected assignments before computing test outcomes.
                atomic_text(OUT/'frozen_choices'/f'{case}.json',json.dumps(dict(private=private.tolist(),group=group.tolist(),selection='validation only',validation_group_losses=val)))
                def loss(c):return np.mean((P[np.arange(S),c][:,TE].mean(0)-truth[TE])**2)
                # Positive gain means lower test aggregate MSE under the population-selected assignment.
                a=loss(private);b=loss(group);margins=[]
                # A positive test margin confirms the private validation preference on held-out data.
                for s in range(S):
                    t=((P[s,:,TE]-truth[TE])**2).mean(1);margin=float(t[1-private[s]]-t[private[s]]);margins.append(margin)
                    MARGINS.append(dict(seed=seed,learner=kind,rho=rho,gap=gap,slot=s,chosen=int(private[s]),test_margin=margin))
                def H(c):return sum((np.mean(c==j))**2 for j in range(2))
                CHOICE.append(dict(seed=seed,learner=kind,rho=rho,gap=gap,private=private.tolist(),group=group.tolist(),
                   private_mse=a,group_mse=b,gain=a-b,relative_gain=(a-b)/a if a>1e-15 else np.nan,
                   private_H=H(private),group_H=H(group),private_concentrated=len(set(private))==1,
                   group_diversified=len(set(group))>1,all_test_margins_positive=min(margins)>0,
                   joint_pattern=(len(set(private))==1 and len(set(group))>1 and min(margins)>0 and a>b),
                   teacher0_mse=np.mean((teachers[0,TE]-truth[TE])**2),teacher1_mse=np.mean((teachers[1,TE]-truth[TE])**2),
                   measured_teacher_corr=np.corrcoef(teachers[:,TE]-truth[TE])[0,1]))
                # Balanced reuse holds each slot-teacher model's usage frequency fixed across collections.
                same=[(0,)*S,(1,)*S];mixed=[c for c in configs if sum(c)==S//2]
                usage=lambda cs:np.array([[np.mean([c[s]==j for c in cs]) for j in range(2)] for s in range(S)])
                assert np.max(abs(usage(same)-usage(mixed)))<1e-12
                ls=np.mean([loss(c) for c in same]);lm=np.mean([loss(c) for c in mixed])
                BALANCED.append(dict(seed=seed,learner=kind,rho=rho,gap=gap,average_concentrated=ls,balanced=lm,gain=ls-lm,
                                    teacher0_concentrated=loss((0,)*S),teacher1_concentrated=loss((1,)*S)))
                atomic_npz(OUT/'choice_predictions'/f'{case}.npz',pred=P,teachers=teachers,truth=truth)
                csv('choice.csv',CHOICE);csv('choice_margins.csv',MARGINS);csv('balanced_choice.csv',BALANCED)
            log(f'Choice {seed} {kind} rho={rho}')


## F. Statistical summaries and figures

The independent unit is the seed. When a metric has several nodes or probes within a seed, those entries are averaged before resampling seeds. Paired comparisons are formed within a seed and then summarized across seeds. Test observations and nodes are not counted as independent training replications.

The full configuration uses 2000 bootstrap resamples of the eight seeds. Intervals are the 2.5th and 97.5th percentiles of bootstrap means. They are descriptive, nominal 95% intervals for individual comparisons. The grid is exploratory, and the intervals have no simultaneous coverage adjustment across tasks, topologies, gaps, or overlaps.

Figures use a consistent color palette and export to 300-dpi PNG and vector PDF. Growth and propagation curves show seed means and seed-bootstrap bands. The individual-versus-population plot shows paired seed-level differences. The teacher-choice heatmaps display every configured cell.

| File stem in `figures/` | What it shows |
|---|---|
| `01_topology_and_loadings` | Four six-node graph examples and theoretical ridge lineage matrices for chain and multiple-parent graphs |
| `02_propagation` | Neural held-out response prediction, update-budget sensitivity, and superposition diagnostics |
| `03_population_growth` | Raw aggregate MSE by population size, learner, and distillation weight |
| `03b_regularization` | Ridge population growth at zero and positive regularization |
| `04_individual_vs_population` | Paired changes in individual and aggregate MSE relative to roots at the largest population size |
| `05_choice_gain` | Mean private-minus-population test aggregate MSE across the complete teacher grid |
| `06_joint_pattern` | Number of seeds satisfying all four choice-pattern conditions, divided by the number of seeds |
| `07_training_curves` | Loss histories for selected nodes of the first seed's longest-budget linear-target chain |

The final convergence figure uses a fixed selection rather than choosing runs by test performance. Theory-based lineage matrices in the first figure use the known ridge coefficients; they are not fitted neural transmission matrices. Publication figures may combine or restyle these results, so manuscript panel numbering is separate from the notebook's export names.

**The next cell** writes these figures without changing model fits or teacher assignments. Human-facing overlap labels use $\gamma$ while input tables retain the field `rho`.

In [ ]:
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.titlesize':12,
 'axes.labelsize':10,'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,
 'grid.alpha':.18,'grid.linewidth':.6,'figure.facecolor':'white','axes.facecolor':'white',
 'savefig.facecolor':'white','pdf.fonttype':42,'ps.fonttype':42})
COLORS=dict(roots='#0072B2',star='#D55E00',chain='#009E73',multi_parent='#CC79A7')
LABELS=dict(roots='Independent roots',star='Star',chain='Chain',multi_parent='Multiple parents')
FIG=OUT/'figures';FIG.mkdir(exist_ok=True)
def savefig(fig,name):
    'Export the same figure to high-resolution PNG and vector PDF, then release it.'
    fig.savefig(FIG/(name+'.png'),dpi=300,bbox_inches='tight');fig.savefig(FIG/(name+'.pdf'),bbox_inches='tight');plt.close(fig)
def ribbon(ax,df,x,y,label,color):
    'Average each x-level within seeds before plotting its seed mean and bootstrap band.'
    rows=[]
    for xx,sub in df.groupby(x):
        values=sub.groupby('seed')[y].mean();mean,lo,hi=bootstrap_mean(values);rows.append((xx,mean,lo,hi))
    ar=np.array(rows,float)
    ax.plot(ar[:,0],ar[:,1],'-o',ms=4,lw=1.8,label=label,color=color)
    ax.fill_between(ar[:,0],ar[:,2],ar[:,3],color=color,alpha=.13,linewidth=0)
    return ar
# Figure 1: six-node examples and loadings from the known ridge benchmark.
fig=plt.figure(figsize=(12,6.5));gs=fig.add_gridspec(2,4,height_ratios=[1,1.3])
for j,topo in enumerate(TOPO):
    ax=fig.add_subplot(gs[0,j]);W=parents(topo,6);pos={i:(i%3,-(i//3)) for i in range(6)}
    if topo=='star':pos={0:(1,.25),**{i:(-.2+(i-1)*.6,-1) for i in range(1,6)}}
    for i in range(6):
        for p in np.where(W[i]>0)[0]:
            ax.annotate('',xy=pos[i],xytext=pos[p],arrowprops=dict(arrowstyle='->',color=COLORS[topo],lw=1.2,shrinkA=14,shrinkB=14,connectionstyle=('arc3,rad=0.45' if abs(pos[i][0]-pos[p][0])>1.5 and pos[i][1]==pos[p][1] else 'arc3,rad=0.10')))
    ax.scatter(*zip(*pos.values()),s=300,color=COLORS[topo],zorder=3)
    for i,(x,y) in pos.items():ax.text(x,y,str(i+1),ha='center',va='center',color='white',weight='bold')
    ax.set(xlim=(-.5,2.5),ylim=(-1.5,.5),title=LABELS[topo]);ax.axis('off')
for j,topo in enumerate(['chain','multi_parent']):
    ax=fig.add_subplot(gs[1,j*2:(j+1)*2]);W=parents(topo,8);Phi=np.linalg.inv(np.eye(8)-CFG['beta']/(1+CFG['ridge'])*W)
    im=ax.imshow(Phi,cmap='Blues',vmin=0,vmax=1);fig.colorbar(im,ax=ax,shrink=.8,label='Source loading')
    ax.set(title=LABELS[topo]+' | theoretical Phi',xlabel='Source node',ylabel='Descendant node');ax.grid(False)
fig.subplots_adjust(hspace=.5,wspace=.65);savefig(fig,'01_topology_and_loadings')
# Figure 2: average held-out probes within each seed before bootstrapping.
pd_=pd.DataFrame(PROP);sd=pd.DataFrame(SUPER);fig,axes=plt.subplots(2,2,figsize=(11,8))
for ax,task in zip(axes[0],['linear','nonlinear']):
    q=pd_[(pd_.learner=='mlp')&(pd_.task==task)&(pd_.budget==LONG)]
    for topo in TOPO[1:]:ribbon(ax,q[q.topology==topo],'node','relative_rmse',LABELS[topo],COLORS[topo])
    ax.set(title=f'MLP | {task} target',xlabel='Node index (root = 0)',ylabel='Relative prediction RMSE')
axes[0,0].legend(fontsize=8)
q=pd_[(pd_.learner=='mlp')&(pd_.task=='linear')&(pd_.topology=='chain')]
ribbon(axes[1,0],q,'budget','relative_rmse','Chain','#009E73')
axes[1,0].set(title='Training budget sensitivity',xlabel='Updates per model',ylabel='Relative prediction RMSE')
for task,color in [('linear','#0072B2'),('nonlinear','#CC79A7')]:
    q=sd[(sd.learner=='mlp')&(sd.task==task)&(sd.test=='superposition')]
    ribbon(axes[1,1],q,'budget','relative_rmse',task,color)
axes[1,1].set(title='Superposition diagnostic',xlabel='Updates per model',ylabel='Relative superposition RMSE');axes[1,1].legend()
fig.tight_layout();savefig(fig,'02_propagation')
# Figure 3: raw aggregate MSE; show unregularized and regularized ridge separately.
gd=pd.DataFrame(GROW);fig,axes=plt.subplots(2,2,figsize=(11,8))
for row,kind in enumerate(LEARNERS):
    for col,beta in enumerate(CFG['betas']):
        ax=axes[row,col];q=gd[(gd.learner==kind)&(gd.beta==beta)&(gd.lam==0)]
        for topo in TOPO:ribbon(ax,q[q.topology==topo],'M','aggregate_mse',LABELS[topo],COLORS[topo])
        ax.set(title=f'{kind.upper()} | beta={beta}',xlabel='Population size',ylabel='Raw aggregate MSE',xticks=CFG['growth_sizes'])
axes[0,0].legend(fontsize=8);fig.tight_layout();savefig(fig,'03_population_growth')
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,lam in zip(axes,[0.,CFG['ridge']]):
    q=gd[(gd.learner=='ridge')&(gd.beta==CFG['beta'])&(gd.lam==lam)]
    for topo in TOPO:ribbon(ax,q[q.topology==topo],'M','aggregate_mse',LABELS[topo],COLORS[topo])
    ax.set(title=f'Ridge regularization lambda={lam}',xlabel='Population size',ylabel='Raw aggregate MSE')
axes[0].legend(fontsize=8);fig.tight_layout();savefig(fig,'03b_regularization')
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,kind in zip(axes,LEARNERS):
    q=pd.DataFrame(PAIRS);q=q[(q.learner==kind)&(q.beta==CFG['beta'])&(q.lam==0)&(q.M==max(CFG['growth_sizes']))]
    for topo in TOPO[1:]:
        t=q[q.topology==topo];ax.scatter(t.individual_difference,t.aggregate_difference,s=35,alpha=.75,color=COLORS[topo],label=LABELS[topo])
    ax.axhline(0,color='#666666',lw=.8);ax.axvline(0,color='#666666',lw=.8)
    ax.set(title=kind.upper()+' | paired vs independent roots',xlabel='Change in individual MSE',ylabel='Change in aggregate MSE')
axes[0].legend(fontsize=8);fig.tight_layout();savefig(fig,'04_individual_vs_population')
# Figures 5 and 6: display every configured grid cell.
cd=pd.DataFrame(CHOICE);lim=max(abs(cd.groupby(['learner','rho','gap']).gain.mean()).max(),1e-10)
for metric,name,title in [('gain','05_choice_gain','Private minus group test MSE'),('joint_pattern','06_joint_pattern','Fraction of seeds showing full pattern')]:
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for ax,kind in zip(axes,LEARNERS):
        table=cd[cd.learner==kind].pivot_table(index='rho',columns='gap',values=metric,aggfunc='mean').reindex(index=CFG['rhos'],columns=CFG['quality_gaps'])
        mat=table.to_numpy(dtype=float)
        im=ax.imshow(mat,aspect='auto',origin='lower',cmap='RdBu_r' if metric=='gain' else 'Blues',vmin=-lim if metric=='gain' else 0,vmax=lim if metric=='gain' else 1)
        ax.set(xticks=np.arange(len(table.columns)),xticklabels=[f'{x:g}' for x in table.columns],yticks=np.arange(len(table.index)),yticklabels=[f'{x:g}' for x in table.index],xlabel='Teacher error amplitude gap',ylabel='Direction overlap gamma',title=kind.upper());ax.grid(False)
        for (i,j),v in np.ndenumerate(mat):
            txt=f'{v:.3g}' if metric=='gain' else f'{int(round(v*len(CFG["seeds"])) )}/{len(CFG["seeds"])}'
            ax.text(j,i,txt,ha='center',va='center',fontsize=8,color='white' if abs(v)>(.65*lim if metric=='gain' else .6) else '#222222')
    fig.colorbar(im,ax=list(axes),shrink=.8,label=title);fig.suptitle(title,y=1.02);savefig(fig,name)
# Figure 7: fixed first seed, chain, baseline arm, and longest budget.
seed=CFG['seeds'][0];P,y,W,loss=train_graph(seed,'mlp','chain',CFG['prop_nodes'],LONG,CFG['beta'],CFG['ridge'],'linear','base')
case=dict(seed=seed,learner='mlp',topology='chain',M=CFG['prop_nodes'],budget=LONG,beta=CFG['beta'],lam=CFG['ridge'],task='linear',variant='base')
ckey=hashlib.sha256(json.dumps(case,sort_keys=True).encode()).hexdigest()[:20]
with np.load(OUT/'graph_predictions'/f'{ckey}.npz',allow_pickle=False) as z:ids=z['model_ids']
fig,ax=plt.subplots(figsize=(7,4))
for i in sorted(set([0,len(ids)//2,len(ids)-1])):
    with np.load(OUT/'model_cache'/str(ids[i])/'completed.npz') as z:h=z['history']
    ax.plot(h[:,0],h[:,1],label=f'Node {i}',lw=1.8)
ax.set(yscale='log',xlabel='Training updates',ylabel='Training MSE',title='Convergence diagnostic | fixed first seed');ax.legend();fig.tight_layout();savefig(fig,'07_training_curves')
print('Figures saved to',FIG)


## G. Summary tables, interpretation, and export

The final cell summarizes teacher-choice gains by learner and grid cell, scans saved neural histories for convergence diagnostics, writes an English results guide, and creates the full archive. It does not select a subset of favorable cells.

### Reading the results

Start with `ridge_identity.csv` to inspect the linear baseline implementation. Then use `training.csv` and `convergence.csv` to assess the fixed-budget fits. Read `propagation.csv` and `superposition.csv` to assess transfer of fitted loadings. Finally, compare raw growth outcomes and the validation-frozen teacher assignments. Numerical agreement with a training identity is distinct from empirical accuracy of that identity as an approximation for a nonlinear learner.

| Output | Contents and interpretation |
|---|---|
| `protocol.json` | Configuration, computational-version identifier, Python version, and NumPy version |
| `data/<seed>.npz` | All inputs, truth coefficients, local directions, intervention directions, and split indices |
| `model_cache/<signature>/` | Completed fits and resumable neural checkpoints |
| `graph_predictions/` | Predictions for all splits, target values, mixture matrices, and case metadata |
| `ridge_identity.csv` | Maximum absolute baseline error-identity discrepancy by seed and topology |
| `propagation.csv` | Fitted loading and held-out prediction metrics for each descendant and probe |
| `superposition.csv` | Superposition and sign-reversal diagnostics |
| `training.csv` | Final training MSE for each propagation case and node |
| `convergence.csv` | Cached neural fits' final losses and relative decrease over their last saved interval |
| `growth.csv` | Individual and aggregate metrics at each population size, with ridge exposure diagnostics |
| `growth_paired_contrasts.csv` | Within-seed changes relative to roots |
| `frozen_choices/` | Validation-selected assignments and validation losses for all candidate allocations |
| `choice_predictions/` | Complete student bank, teacher predictions, and truth for each choice case |
| `choice.csv` | Seed-level private and population test MSE, gains, concentration, and joint-pattern flags |
| `choice_margins.csv` | Each student's held-out margin for its validation-selected teacher |
| `balanced_choice.csv` | Matched-use comparisons between average concentrated and balanced assignments |
| `choice_seed_summary.csv` | Mean gain, pointwise bootstrap interval, seed count, and pattern counts |
| `READ_ME_RESULTS.md` | Automatically generated English report based on the computed tables |
| `COMPLETE.json` | Completion marker for the analysis stage, written before ZIP creation |

`relative_gain` in `choice.csv` is a within-seed ratio. A reported percentage computed from mean private and mean population MSE is a ratio of means and can differ from the mean of those seed-level ratios. The summary reports mean paired absolute gains and their bootstrap intervals.

A sustained final-interval loss decrease indicates a fixed-budget fit that may benefit from further optimization. The interval compares the last two saved losses and is not a formal convergence criterion. The full report includes the nonlinear approximation diagnostics and the regularized ridge comparison alongside the main settings.

### Finding and recovering the files

The active path is printed as `OUT`. In the retained original run it was `/content/drive/MyDrive/lineage_dag_full/b23c247b3a094046`. The full ZIP is written beside that folder as `<run_key>_full_results.zip`. A changed configuration or environment may produce a different key.

The archive includes predictions and caches, so its size can greatly exceed the CSV tables and figures. In Colab, the final cell also requests a browser download. If the browser does not deliver it, retrieve the already saved ZIP from Drive. The notebook itself contains the retained execution log and summary table, but not the external model-cache files needed for resuming the original run.

In [ ]:
# Each group contains one paired gain per independent seed; retain every teacher-grid cell.
SUM=[]
for keys,g in cd.groupby(['learner','rho','gap']):
    mean,lo,hi=bootstrap_mean(g.gain)
    SUM.append(dict(learner=keys[0],rho=keys[1],gap=keys[2],mean_gain=mean,ci_low=lo,ci_high=hi,
                    n_seeds=len(g),joint_pattern_count=int(g.joint_pattern.sum()),private_concentrated_count=int(g.private_concentrated.sum())))
csv('choice_seed_summary.csv',SUM)
# The last saved-interval loss decrease diagnoses finite-budget training, not formal convergence.
CONVERGENCE=[]
for path in sorted((OUT/'model_cache').glob('*/completed.npz')):
    with np.load(path,allow_pickle=False) as z:
        if 'history' not in z:continue
        h=z['history'];ratio=(h[-2,1]-h[-1,1])/h[-2,1] if len(h)>1 and h[-2,1]>1e-15 else np.nan
        CONVERGENCE.append(dict(model=path.parent.name,steps=int(h[-1,0]),final_loss=float(z['train_mse']),last_interval_relative_drop=ratio))
csv('convergence.csv',CONVERGENCE)
lines=['# Full mechanism-training experiment results',f'Mode {MODE}, independent seeds {len(CFG["seeds"])}.','All data are synthetic regression data; these are not LLM training experiments.',
       'Teacher assignments are selected on validation data and frozen for test evaluation. Loadings are fitted on h1 validation responses and evaluated on h2 and negative-direction test responses.',
       'Budgets and topologies share seeds for paired comparisons. Bootstrap resampling uses seeds, with exploratory pointwise intervals and no whole-grid multiplicity correction.']
for (kind,task,budget),g in pd_[pd_.topology!='roots'].groupby(['learner','task','budget']):
    per_seed=g.groupby('seed').relative_rmse.mean();mean,lo,hi=bootstrap_mean(per_seed)
    lines.append(f'{kind}/{task}/{budget} updates: seed-mean descendant relative RMSE {mean:.4f}, descriptive 95% interval [{lo:.4f}, {hi:.4f}].')
for row in SUM:
    lines.append(f"Choice {row['learner']} rho={row['rho']} gap={row['gap']}: population gain {row['mean_gain']:.6f}, interval [{row['ci_low']:.6f},{row['ci_high']:.6f}], full joint pattern {row['joint_pattern_count']}/{row['n_seeds']} seeds.")
conv=pd.DataFrame(CONVERGENCE);c=conv[conv.steps==LONG]
lines.append(f'Median relative loss decrease over the last saved interval among longest-budget models: {c.last_interval_relative_drop.median():.3%}, a convergence diagnostic only.')
lines+=['Read ridge_identity.csv first, then training/convergence, propagation/superposition, and finally growth and frozen teacher choices.',
        'Continued loss decreases indicate fixed-budget rather than fully converged results. Nonlinear-target prediction errors describe the limits of the approximation.',
        'Compare growth at lambda=0 and lambda>0 to distinguish shared shrinkage from lineage effects.',
        'balanced_choice compares the average of the two concentrated assignments with balanced reuse; it is not a comparison against the best teacher.',
        f'Elapsed minutes {(time.time()-START)/60:.2f}; newly trained models {COUNTS["trained"]}; reused models {COUNTS["reused"]}.']
atomic_text(OUT/'READ_ME_RESULTS.md','\n\n'.join(lines))
atomic_text(OUT/'COMPLETE.json',json.dumps(dict(status='complete',mode=MODE,code=CODE_SHA)))
# Include predictions and resumable caches in the full archive, which can be substantially larger than tables.
archive=OUT.parent/(OUT.name+'_full_results.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and not p.name.endswith('.tmp') and '.corrupt_' not in p.name:z.write(p,p.relative_to(OUT))
print('Original experiment complete. Tables and figures saved.')
if SHOW_ALL_TABLES:display(pd.DataFrame(SUM))
# The ZIP is already saved on Drive; this additionally requests a browser download.
PAPER_OUT=OUT
PAPER_ARCHIVE=archive
# Prefer the source just reproduced; the supplements verify its protocol and data.
if MODE=='full':
    os.environ['LINEAGE_SOURCE']=str(PAPER_OUT)
    os.environ['LINEAGE_SOURCE_ZIP']=str(PAPER_ARCHIVE)
else:
    # Supplement smoke tests require the full-size recorded data, with shortened training.
    os.environ.setdefault('LINEAGE_SOURCE',str(RECORDED_SOURCE))
    os.environ.setdefault('LINEAGE_SOURCE_ZIP',str(RECORDED_ZIP))
    os.environ.setdefault('LINEAGE_OPTIMIZER_MODE','smoke')
    os.environ.setdefault('LINEAGE_INTEGRATED_MODE','smoke')


## H. Optimizer comparison in the original network

This paired comparison retains the original width-16 tanh network, no L2 penalty,
data splits, eight seeds, initialization, learning rate 0.01, and full-batch training.
Only the update rule changes: Adam versus gradient descent without momentum.
The primary comparison uses 600 updates. The 1800-update and chain-amplitude checks
are retained in the saved diagnostic tables. The inline table reports both tasks
at the primary budget; all seeds enter the summary.


In [ ]:
import os,sys,json,time,hashlib,io,zipfile,platform,itertools,importlib.util,subprocess
from pathlib import Path
for k in ['OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS']:os.environ.setdefault(k,'1')
for pkg in ['numpy','pandas','matplotlib','threadpoolctl']:
    if importlib.util.find_spec(pkg) is None:subprocess.check_call([sys.executable,'-m','pip','install',pkg])
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
THREAD_LIMIT=threadpool_limits(limits=1)
try:from IPython.display import display,Markdown
except ImportError:
    display=print;Markdown=str
try:
    import google.colab
    IN_COLAB=True
except ImportError:IN_COLAB=False
if IN_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive')
SOURCE_DIR=Path(os.environ.get('LINEAGE_SOURCE','/content/drive/MyDrive/lineage_dag_full/b23c247b3a094046'))
SOURCE_ZIP=Path(os.environ.get('LINEAGE_SOURCE_ZIP',str(SOURCE_DIR)+'_full_results.zip'))
OUTPUT_ROOT=Path(os.environ.get('LINEAGE_OPTIMIZER_OUTPUT','/content/drive/MyDrive/lineage_optimizer_check' if IN_COLAB else './lineage_optimizer_check'))
MODE=os.environ.get('LINEAGE_OPTIMIZER_MODE','full')
RUN_LONG=RUN_LONG_OPTIMIZER
RUN_AMPLITUDE=RUN_AMPLITUDE_OPTIMIZER
OPTIMIZER_AUTO_DOWNLOAD=False
BUILD_SHA='d722298f5efa59cff55d670013fdd6fb3eb0bc31bfab91d1260715df22c9f2c1'
ORIGINAL_SHA='1ac6b1a53dbbb84842cf1ff69735acec25783f612a8b1cc61a82696fc900d01d'
START=time.time();COUNTS=dict(new_models=0,old_models=0,cached_models=0,resumed_models=0,new_graphs=0,old_graphs=0,cached_graphs=0,updates=0)

def atomic_text(path,s):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);q=path.with_name(path.name+'.tmp');q.write_text(s,encoding='utf-8');os.replace(q,path)
def atomic_npz(path,**obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);q=path.with_name(path.name+'.tmp')
    with open(q,'wb') as f:np.savez_compressed(f,**obj)
    os.replace(q,path)
def read_npz(path):
    with np.load(path,allow_pickle=False) as z:return {k:z[k].copy() for k in z.files}
def digest(o):return hashlib.sha256(json.dumps(o,sort_keys=True,separators=(',',':')).encode()).hexdigest()
def csv(name,rows):
    d=pd.DataFrame(rows);atomic_text(REPORT/name,d.to_csv(index=False));return d
def log(s):print(f'{s} | {(time.time()-START)/60:.1f} min | {COUNTS}',flush=True)


### H1. Recorded data and cache verification
Read the original protocol and exact saved arrays. Full mode uses eight seeds; smoke mode uses the first two recorded seeds and shorter training.

In [ ]:
class Source:
    def __init__(self,directory,archive):
        self.directory=directory;self.z=None;self.prefix='';self.names=set();self.accessed={}
        if archive.is_file():
            self.z=zipfile.ZipFile(archive);self.names=set(self.z.namelist())
            pp=[n for n in self.names if n=='protocol.json' or n.endswith('/protocol.json')]
            if len(pp)!=1:raise ValueError('Expected exactly one protocol.json in the source archive')
            self.prefix=pp[0][:-len('protocol.json')]
        if not self.exists('protocol.json'):raise FileNotFoundError(f'Source data not found: {directory} or {archive}')
        if self.z and (directory/'protocol.json').is_file():
            assert json.loads((directory/'protocol.json').read_text())==json.loads(self.z.read(self.prefix+'protocol.json'))
    def exists(self,rel):return (self.directory/rel).is_file() or self.prefix+rel in self.names
    def read(self,rel):
        p=self.directory/rel
        if p.is_file():b=p.read_bytes()
        elif self.z and self.prefix+rel in self.names:b=self.z.read(self.prefix+rel)
        else:raise FileNotFoundError(rel)
        self.accessed[rel]=hashlib.sha256(b).hexdigest();return b
    def npz(self,rel):
        with np.load(io.BytesIO(self.read(rel)),allow_pickle=False) as z:return {k:z[k].copy() for k in z.files}
SOURCE=Source(SOURCE_DIR,SOURCE_ZIP);PROTOCOL=json.loads(SOURCE.read('protocol.json'));CFG=PROTOCOL['config']
assert PROTOCOL['code_sha']==ORIGINAL_SHA
for k,v in dict(mode='full',d=8,train=192,validation=256,test=1024,nodes=16,prop_nodes=8,hidden=16,beta=.7,delta=.15,learning_rate=.01,ridge=.05).items():
    assert CFG[k]==v,f'Source protocol mismatch: {k}={CFG[k]}'
assert CFG['seeds']==list(range(2701,2709))
if MODE=='full':
    SEEDS=CFG['seeds'];M=8;BASE_STEPS=600;LONG_STEPS=1800;SAVE_EVERY=100;LOG_EVERY=50;BOOT=2000
elif MODE=='smoke':
    SEEDS=CFG['seeds'][:2];M=3;BASE_STEPS=8;LONG_STEPS=16;SAVE_EVERY=4;LOG_EVERY=4;BOOT=100
else:raise ValueError('MODE must be full or smoke')
TASKS=['linear','nonlinear'];TOPOS=['roots','star','chain','multi_parent'];OPTIMIZERS=['adam','gd']
BUDGETS=[BASE_STEPS]+([LONG_STEPS] if RUN_LONG else [])
AMPLITUDES=[1.,.5,.25] if RUN_AMPLITUDE else [1.]
VARIANTS={'base':(0,0),'h1':(1,0),'h2':(0,1),'negative':(-1,0),'sum':(1,1)}
N=CFG['train'];VEND=N+CFG['validation'];TOTAL=VEND+CFG['test'];TR=slice(0,N);VA=slice(N,VEND);TE=slice(VEND,None)
DATA={}
for seed in SEEDS:
    z=SOURCE.npz(f'data/{seed}.npz');X,theta,noise,dirs=[z[k] for k in ['X','theta','noise','dirs']]
    assert X.shape==(TOTAL,8) and theta.shape==(8,) and noise.shape==(16,8) and dirs.shape==(2,8)
    for k,e in [('train_indices',np.arange(N)),('validation_indices',np.arange(N,VEND)),('test_indices',np.arange(VEND,TOTAL))]:np.testing.assert_array_equal(z[k],e)
    for a in [X,theta,noise,dirs]:assert np.isfinite(a).all()
    np.testing.assert_allclose(X[TR].T@X[TR]/N,np.eye(8),atol=1e-10)
    DATA[seed]=(X,theta,noise,dirs)
DATA_HASH=dict(SOURCE.accessed)
CACHE_PLAN=dict(version='optimizer_original_network_v1',build_sha=BUILD_SHA,mode=MODE,source_hashes=DATA_HASH,hidden=16,l2=0.,lr=CFG['learning_rate'],beta=CFG['beta'],delta=CFG['delta'],numpy=np.__version__,python=platform.python_version())
RUN=OUTPUT_ROOT/('optimizer_original_network_v1_'+MODE+'_'+digest(CACHE_PLAN)[:12]);REPORT=RUN/'reports'/'anonymous_english'
assert SOURCE_DIR.resolve()!=RUN.resolve() and SOURCE_DIR.resolve() not in RUN.resolve().parents
REPORT.mkdir(parents=True,exist_ok=True)
atomic_text(RUN/'training_plan.json',json.dumps(CACHE_PLAN,indent=2))
atomic_text(REPORT/'experiment_plan.json',json.dumps(dict(seeds=SEEDS,budgets=BUDGETS,optimizers=OPTIMIZERS,tasks=TASKS,topologies=TOPOS,M=M,amplitudes=AMPLITUDES,source_dir=str(SOURCE_DIR),primary_steps=BASE_STEPS),indent=2))
if (REPORT/'COMPLETE.json').exists():(REPORT/'COMPLETE.json').unlink()
print('Source: ',SOURCE_DIR,'\nOutput: ',RUN,'\nMode: ',MODE,'Budgets: ',BUDGETS,'Chain amplitudes: ',AMPLITUDES)


### H2. Paired update rules

$f_\theta(x)=v^\top\tanh(U^\top x+b)+c$ and
$L=n^{-1}\sum_i(f_\theta(x_i)-y_i)^2$.
GD uses $\theta_{t+1}=\theta_t-0.01\nabla L_t$.
Adam retains $(\beta_1,\beta_2,\epsilon)=(0.9,0.999,10^{-8})$ and bias correction.
Every parameter is trained. Cache keys include the optimizer, target, initialization,
and learning rate. Numerical update expressions match the separate optimizer notebook.

In [ ]:
def truth_fn(X,theta,task):return X@theta+(0.3*np.sin(X[:,0]*X[:,1]) if task=='nonlinear' else 0.)
def parents(topo,m):
    W=np.zeros((m,m))
    if topo=='star':W[1:,0]=1
    elif topo=='chain':W[np.arange(1,m),np.arange(m-1)]=1
    elif topo=='multi_parent':
        for i in range(1,m):
            js=sorted(set([0,i-1]));W[i,js]=1/len(js)
    elif topo!='roots':raise ValueError(topo)
    return W

def init_net(seed):
    'Initialize a one-hidden-layer tanh network reproducibly for one student slot.'
    r=np.random.default_rng(seed);d=CFG['d'];h=CFG['hidden']
    return [r.normal(size=(d,h))/np.sqrt(d),np.zeros(h),r.normal(size=h)/np.sqrt(h),np.zeros(1)]
def forward(X,p):
    'Return scalar predictions and hidden activations for all supplied rows.'
    H=np.tanh(X@p[0]+p[1]);return H@p[2]+p[3][0],H
def objective_grad(X,y,p):
    'Compute full-batch mean squared loss and analytic gradients for all four parameter arrays.'
    pred,H=forward(X,p);err=pred-y;n=len(y)
    g=2*err/n;dh=(g[:,None]*p[2])*(1-H*H)
    return float(np.mean(err**2)),[X.T@dh,dh.sum(0),H.T@g,np.array([g.sum()])]
def update(p,m,v,g,step,optimizer,lr):
    for k in range(4):
        if optimizer=='adam':
            m[k]=.9*m[k]+.1*g[k];v[k]=.999*v[k]+.001*g[k]**2
            p[k]-=lr*(m[k]/(1-.9**step))/(np.sqrt(v[k]/(1-.999**step))+1e-8)
        elif optimizer=='gd':p[k]-=lr*g[k]
        else:raise ValueError(optimizer)

def model_key(X,target,seed,opt):
    meta=dict(build_sha=BUILD_SHA,seed=int(seed),optimizer=opt,hidden=CFG['hidden'],lr=CFG['learning_rate'])
    return hashlib.sha256(X.tobytes()+target[TR].tobytes()+json.dumps(meta,sort_keys=True).encode()).hexdigest()

def original_model(X,target,seed,steps):
    if MODE!='full':return None
    for lam in [.05,0.]:
        sig=hashlib.sha256(X.tobytes()+target[TR].tobytes()+str((int(seed),'mlp',int(steps),lam,ORIGINAL_SHA)).encode()).hexdigest()
        rel=f'model_cache/{sig}/completed.npz'
        if SOURCE.exists(rel):
            z=SOURCE.npz(rel);assert str(z['signature'])==sig
            assert z['pred'].shape==(TOTAL,) and np.isfinite(z['pred']).all()
            return z,sig
    return None

def train_model(X,target,seed,opt,steps,allow_old=True,interrupt_at=None,cache_salt=''):
    sig=model_key(X,target,seed,opt)+cache_salt;folder=RUN/'model_cache'/sig
    done=folder/f'completed_{steps}.npz';checkpoint=folder/'checkpoint.npz'
    if done.is_file():
        z=read_npz(done);assert str(z['signature'])==sig;COUNTS['cached_models']+=1;return z,sig
    if opt=='adam' and allow_old:
        old=original_model(X,target,seed,steps)
        if old:
            z,osig=old;z['signature']=sig;z['origin']='original:'+osig
            atomic_npz(done,**z);COUNTS['old_models']+=1;return z,sig
    p=init_net(seed);m=[np.zeros_like(a) for a in p];v=[np.zeros_like(a) for a in p];start=0;history=[]
    if checkpoint.is_file():
        z=read_npz(checkpoint);assert str(z['signature'])==sig
        if int(z['step'])<=steps:
            start=int(z['step']);p=[z[f'p{k}'] for k in range(4)];m=[z[f'm{k}'] for k in range(4)];v=[z[f'v{k}'] for k in range(4)]
            history=z['history'].tolist();COUNTS['resumed_models']+=1
    def pack(t):
        z=dict(signature=sig,step=t,history=np.array(history,float),origin='new',optimizer=opt)
        for k in range(4):z.update({f'p{k}':p[k],f'm{k}':m[k],f'v{k}':v[k]})
        return z
    if start==0:
        loss,g=objective_grad(X[TR],target[TR],p);history=[[0,loss,float(np.sqrt(sum(np.sum(a*a) for a in g)))]]
    for step in range(start+1,steps+1):
        loss,g=objective_grad(X[TR],target[TR],p)
        if not np.isfinite(loss) or not all(np.isfinite(a).all() for a in g):raise FloatingPointError(f'{opt}, model={sig}, step={step}')
        update(p,m,v,g,step,opt,CFG['learning_rate']);COUNTS['updates']+=1
        if step%LOG_EVERY==0 or step==steps or step==interrupt_at:
            loss,g=objective_grad(X[TR],target[TR],p);history.append([step,loss,float(np.sqrt(sum(np.sum(a*a) for a in g)))])
        if step%SAVE_EVERY==0 or step==steps or step==interrupt_at:atomic_npz(checkpoint,**pack(step))
        if step==interrupt_at:raise InterruptedError('Test checkpoint saved')
    z=pack(steps);z['pred']=forward(X,p)[0];z['train_mse']=float(np.mean((z['pred'][TR]-target[TR])**2))
    assert np.isfinite(z['pred']).all();atomic_npz(done,**z);COUNTS['new_models']+=1
    return z,sig

def local_target(X,y,noise,dirs,i,variant,amplitude):
    # Preserve the original split-wise numerical operation order.
    out=np.empty(TOTAL);a,b=VARIANTS[variant]
    for sl in [TR,VA,TE]:
        out[sl]=y[sl]+X[sl]@noise[i]
        if i==0:out[sl]=out[sl]+(CFG['delta']*amplitude)*X[sl]@(a*dirs[0]+b*dirs[1])
    return out

def node_target(X,y,noise,dirs,W,P,i,variant,amplitude):
    local=local_target(X,y,noise,dirs,i,variant,amplitude)
    return local if not W[i].sum() else (1-CFG['beta'])*local+CFG['beta']*(W[i,:i]@np.array(P))

def graph(seed,task,topo,opt,steps,variant,amplitude=1.,audit_zero=False):
    if variant=='base':amplitude=1.  # All perturbation amplitudes share the same baseline.
    case=dict(seed=int(seed),task=task,topology=topo,optimizer=opt,steps=int(steps),variant=variant,amplitude=float(amplitude),M=M,audit_zero=audit_zero)
    path=RUN/'graph_predictions'/(digest(case)[:24]+'.npz')
    if path.is_file():
        z=read_npz(path);assert json.loads(str(z['case_json']))==case;COUNTS['cached_graphs']+=1;return z
    X,theta,noise,dirs=DATA[seed];y=truth_fn(X,theta,task);W=parents(topo,M)
    if opt=='adam' and amplitude==1. and MODE=='full' and not audit_zero:
        original=dict(seed=int(seed),learner='mlp',topology=topo,M=M,budget=int(steps),beta=float(CFG['beta']),lam=.05,task=task,variant=variant)
        key=hashlib.sha256(json.dumps(original,sort_keys=True).encode()).hexdigest()[:20];rel=f'graph_predictions/{key}.npz'
        if SOURCE.exists(rel):
            z=SOURCE.npz(rel);assert json.loads(str(z['case_json']))==original
            np.testing.assert_array_equal(z['W'],W);np.testing.assert_allclose(z['truth'],y,atol=1e-12,rtol=1e-12)
            assert z['pred'].shape==(M,TOTAL) and np.isfinite(z['pred']).all()
            z.update(case_json=json.dumps(case),origin='original_graph')
            atomic_npz(path,**z);COUNTS['old_graphs']+=1;return z
    P=[];ids=[];losses=[]
    for i in range(M):
        target=node_target(X,y,noise,dirs,W,P,i,variant,amplitude)
        z,sig=train_model(X,target,seed*100+i,opt,steps)
        P.append(z['pred']);ids.append(sig);losses.append(z['train_mse'])
    z=dict(pred=np.array(P),truth=y,W=W,model_ids=np.array(ids),train_mse=np.array(losses),case_json=json.dumps(case),origin='new')
    atomic_npz(path,**z);COUNTS['new_graphs']+=1;return z


### H3. Response metrics

Fit each edge loading to the h1 validation response; freeze the loadings and predict
h2 and negative-direction test responses, conditional on the observed root response.
The primary metric averages node/probe relative RMSE within each seed and then across seeds.
The original $10^{-16}$ cutoff is retained. Raw energies, one-step and recursive
predictions, nominal-loadings predictions, superposition, zero-intervention controls,
and task-fit diagnostics are also saved. Pooled energy metrics use summed squared errors
and summed response energy, a distinct aggregation from the primary metric.

In [ ]:
ROWS=[];LOADINGS=[];FIT=[];SUPER=[];AMPLITUDE=[];CONTROLS=[];AUDIT=[]
def recursive(obs,A,W,one_step=False):
    hat=np.zeros_like(obs);hat[0]=obs[0]
    for i in range(1,len(A)):
        js=np.flatnonzero(W[i])
        if len(js):hat[i]=A[i,js]@(obs[js] if one_step else hat[js])
    return hat
def calibrate(D,W):
    A=np.zeros_like(W);denoms=[]
    for i in range(1,len(W)):
        if not W[i].sum():denoms.append(0.);continue
        z=W[i]@D[:,VA];den=float(z@z);denoms.append(den)
        A[i]=(float(z@D[i,VA])/den)*W[i] if den>1e-16 else np.nan
    return A,denoms
def evaluate(seed,task,topo,opt,steps,amplitude,bank,reference=None):
    W=bank['base']['W']
    P=bank['base']['pred'];y=bank['base']['truth'];D={v:bank[v]['pred']-P for v in VARIANTS if v!='base'}
    A,den=calibrate(D['h1'],W);depth=np.zeros(M,int)
    for i in range(1,M):
        js=np.flatnonzero(W[i]);depth[i]=1+max(depth[js]) if len(js) else 0
    meta=dict(seed=seed,task=task,topology=topo,optimizer=opt,steps=steps,amplitude=amplitude)
    for i in range(1,M):
        LOADINGS.append(dict(**meta,node=i,depth=int(depth[i]),loading=float(A[i].sum()),parent_calibration_energy=den[i-1]/CFG['validation'],identified=bool(W[i].sum() and np.isfinite(A[i]).all())))
    Aref=calibrate(reference['h1'],W)[0] if reference is not None else A
    for probe in ['h2','negative']:
        obs=D[probe][:,TE]
        hats={'calibrated_recursive':recursive(obs,A,W),'calibrated_one_step':recursive(obs,A,W,True),'nominal_beta':recursive(obs,CFG['beta']*W,W)}
        if amplitude!=1.:hats['fixed_original_A']=recursive(obs,Aref,W)
        for method,hat in hats.items():
            for i in range(1,M):
                error=float(np.mean((obs[i]-hat[i])**2));energy=float(np.mean(obs[i]**2));parent=W[i]@obs
                pe=float(np.mean(parent**2));cos=float(np.mean(parent*obs[i])/np.sqrt(pe*energy)) if pe*energy>1e-32 else np.nan
                ROWS.append(dict(**meta,probe=probe,method=method,node=i,depth=int(depth[i]),error_energy=error,response_energy=energy,
                                 rmse=np.sqrt(error),response_rms=np.sqrt(energy),relative_rmse=np.sqrt(error/energy) if energy>1e-16 else np.nan,
                                 energy_above_original_cutoff=energy>1e-16,parent_response_rms=np.sqrt(pe),parent_child_cosine=cos,
                                 root_response_rms=float(np.sqrt(np.mean(obs[0]**2)))))
        if topo=='roots':CONTROLS.append(dict(**meta,probe=probe,check='independent_roots',max_abs=float(np.max(np.abs(obs[1:])))))
        if reference is not None:
            expected=amplitude*reference[probe][:,TE]
            for i in range(M):
                err=float(np.mean((obs[i]-expected[i])**2));en=float(np.mean(expected[i]**2))
                AMPLITUDE.append(dict(**meta,probe=probe,node=i,depth=int(depth[i]),actual_response_rms=float(np.sqrt(np.mean(obs[i]**2))),response_rms_per_delta=float(np.sqrt(np.mean(obs[i]**2))/(CFG['delta']*amplitude)),
                                      homogeneity_rmse=np.sqrt(err),expected_response_energy=en,homogeneity_relative_rmse=np.sqrt(err/en) if en>1e-16 else np.nan))
    for test,res,ref in [('superposition',D['sum']-D['h1']-D['h2'],D['h1']+D['h2']),('sign',D['negative']+D['h1'],D['h1'])]:
        num=float(np.mean(res[:,TE]**2));denom=float(np.mean(ref[:,TE]**2))
        SUPER.append(dict(**meta,test=test,error_energy=num,response_energy=denom,relative_rmse=np.sqrt(num/denom) if denom>1e-16 else np.nan))
    if amplitude==1.:
        X,theta,noise,dirs=DATA[seed]
        for variant,z in bank.items():
            PP=z['pred']
            for i in range(M):
                target=node_target(X,y,noise,dirs,W,PP[:i],i,variant,1.)
                FIT.append(dict(**meta,variant=variant,node=i,depth=int(depth[i]),train_target_mse=float(np.mean((PP[i,TR]-target[TR])**2)),
                                validation_target_mse=float(np.mean((PP[i,VA]-target[VA])**2)),truth_test_mse=float(np.mean((PP[i,TE]-y[TE])**2)),
                                aggregate_truth_test_mse=float(np.mean((PP[:,TE].mean(0)-y[TE])**2))))
    return D

def bootstrap(a):
    a=np.array(a,float);a=a[np.isfinite(a)]
    if not len(a):return np.nan,np.nan,np.nan
    if len(a)==1:return float(a[0]),np.nan,np.nan
    means=a[np.random.default_rng(20261108).integers(0,len(a),(BOOT,len(a)))].mean(1)
    return float(a.mean()),float(np.quantile(means,.025)),float(np.quantile(means,.975))
def summary(df,keys,metrics):
    out=[]
    for key,g in df.groupby(keys,dropna=False):
        if not isinstance(key,tuple):key=(key,)
        for metric in metrics:
            a=g.groupby('seed')[metric].mean();mean,lo,hi=bootstrap(a)
            out.append(dict(zip(keys,key),metric=metric,mean=mean,ci_low=lo,ci_high=hi,n_seeds=int(a.notna().sum())))
    return pd.DataFrame(out)
def paired(df,keys,metrics):
    out=[]
    for metric in metrics:
        d=df.groupby(keys+['seed','optimizer'],dropna=False)[metric].mean().unstack('optimizer')
        if not {'adam','gd'}.issubset(d.columns):continue
        for key,g in d.reset_index().groupby(keys,dropna=False):
            if not isinstance(key,tuple):key=(key,)
            valid=g[['adam','gd']].dropna();diff=valid.gd-valid.adam;mean,lo,hi=bootstrap(diff)
            out.append(dict(zip(keys,key),metric=metric,adam_mean=float(valid.adam.mean()),gd_mean=float(valid.gd.mean()),gd_minus_adam=mean,
                            ci_low=lo,ci_high=hi,n_pairs=len(valid),gd_lower_seeds=int((diff<0).sum()),
                            reduction_pct_of_means=100*(1-valid.gd.mean()/valid.adam.mean()) if abs(valid.adam.mean())>1e-16 else np.nan))
    return pd.DataFrame(out)

RAW={'node_response.csv':ROWS,'loadings.csv':LOADINGS,'task_fit.csv':FIT,'superposition_sign.csv':SUPER,'amplitude.csv':AMPLITUDE,'controls.csv':CONTROLS,'original_reproduction.csv':AUDIT}
def save_raw():
    for name,rows in RAW.items():
        if rows:csv(name,rows)
    atomic_text(REPORT/'progress.json',json.dumps(dict(elapsed_minutes=(time.time()-START)/60,counters=COUNTS),indent=2))


### H4. Run the paired experiments
Run numerical checks, then the configured budgets and amplitudes. Checkpoints are retained; nonfinite computations raise an error.

In [ ]:
# Gradient and update checks; all training randomness is paired.
p=init_net(42);r=np.random.default_rng(91);xx=r.normal(size=(11,8));yy=r.normal(size=11);_,g=objective_grad(xx,yy,p)
for k,idx in [(0,(1,2)),(1,(2,)),(2,(3,)),(3,(0,))]:
    old=p[k][idx];eps=1e-6;p[k][idx]=old+eps;hi=objective_grad(xx,yy,p)[0];p[k][idx]=old-eps;lo=objective_grad(xx,yy,p)[0];p[k][idx]=old
    assert abs((hi-lo)/(2*eps)-g[k][idx])<1e-6
print('Original-network gradient check passed; Adam and GD share initialization.')

# Independently reproduce available original root predictions.
if MODE=='full':
    seed=SEEDS[0];X,theta,noise,dirs=DATA[seed]
    for task in TASKS:
        y=truth_fn(X,theta,task);target=local_target(X,y,noise,dirs,0,'base',1.)
        old=original_model(X,target,seed*100,BASE_STEPS)
        if old:
            fresh,_=train_model(X,target,seed*100,'adam',BASE_STEPS,allow_old=False,cache_salt='_fresh_audit')
            relative=float(np.linalg.norm(fresh['pred']-old[0]['pred'])/max(np.linalg.norm(old[0]['pred']),1e-16))
            AUDIT.append(dict(seed=seed,task=task,check='fresh_Adam_root_vs_original',relative_difference=relative))
            if relative>1e-6:raise RuntimeError('Adam root reproduction failed; check the numerical environment. Original caches are preserved.')
        else:AUDIT.append(dict(seed=seed,task=task,check='fresh_Adam_root_vs_original',relative_difference=np.nan,reason='Original root cache unavailable; train the missing model'))
for steps in BUDGETS:
    for seed in SEEDS:
        for task in TASKS:
            for opt in OPTIMIZERS:
                for topo in TOPOS:
                    bank={v:graph(seed,task,topo,opt,steps,v) for v in VARIANTS}
                    D=evaluate(seed,task,topo,opt,steps,1.,bank)
                    if topo=='chain' and steps==BASE_STEPS:
                        zero=graph(seed,task,topo,opt,steps,'h1',0.,audit_zero=True)
                        zero_base=graph(seed,task,topo,opt,steps,'base',audit_zero=True)
                        zero_gap=float(np.max(np.abs(zero['pred']-zero_base['pred'])))
                        CONTROLS.append(dict(seed=seed,task=task,topology=topo,optimizer=opt,steps=steps,amplitude=0.,probe='h1',check='zero_intervention',max_abs=zero_gap))
                        assert zero_gap<1e-12,'Nonzero response under a zero intervention'
                        for amplitude in AMPLITUDES[1:]:
                            small={v:graph(seed,task,topo,opt,steps,v,amplitude) for v in VARIANTS}
                            evaluate(seed,task,topo,opt,steps,amplitude,small,reference=D)
                save_raw()
                print(f'{task} seed={seed} {opt} steps={steps} | {(time.time()-START)/60:.1f} min | {COUNTS}',flush=True)
save_raw()
print('All configured optimizer experiments completed.')


### H5. Summarize and export
Seeds are the independent statistical units. Save all diagnostic tables and pointwise paired bootstrap intervals; display the primary 600-update response comparison.

In [ ]:
node=pd.DataFrame(ROWS);loading=pd.DataFrame(LOADINGS);fit=pd.DataFrame(FIT);sup=pd.DataFrame(SUPER)
# Save root controls separately; summarize ratios on graphs with edges.
d=node[node.topology!='roots'].copy()
condition=['task','optimizer','steps','amplitude','method']
seed_rows=[]
for key,g in d.groupby(condition+['seed'],dropna=False):
    good=np.isfinite(g.error_energy)&np.isfinite(g.response_energy)
    numerator=float(g.loc[good,'error_energy'].sum());denominator=float(g.loc[good,'response_energy'].sum())
    seed_rows.append(dict(zip(condition+['seed'],key),mean_node_relative_rmse=float(g.relative_rmse.mean()),
                          pooled_relative_rmse=np.sqrt(numerator/denominator) if denominator>0 else np.nan,
                          pooled_energy_skill=1-numerator/denominator if denominator>0 else np.nan,
                          absolute_rmse=np.sqrt(float(g.error_energy.mean())),response_rms=np.sqrt(float(g.response_energy.mean())),
                          n_nodes_probes=len(g),n_finite_predictions=int(good.sum()),n_above_original_cutoff=int(g.energy_above_original_cutoff.sum())))
seed_metrics=csv('seed_response_metrics.csv',seed_rows)
metric_cols=['mean_node_relative_rmse','pooled_relative_rmse','pooled_energy_skill','absolute_rmse','response_rms']
csv('response_summary.csv',summary(seed_metrics,condition,metric_cols))
primary=seed_metrics[(seed_metrics.steps==BASE_STEPS)&(seed_metrics.amplitude==1.)&(seed_metrics.method=='calibrated_recursive')]
primary_compare=csv('primary_Adam_GD_paired.csv',paired(primary,['task'],metric_cols))
csv('budget_Adam_GD_paired.csv',paired(seed_metrics[seed_metrics.amplitude==1.],['task','steps','method'],metric_cols))
csv('depth_summary.csv',summary(d,['task','topology','optimizer','steps','amplitude','method','depth'],['relative_rmse','rmse','response_rms','parent_child_cosine','parent_response_rms']))
csv('loading_summary.csv',summary(loading[loading.topology!='roots'],['task','topology','optimizer','steps','amplitude','depth'],['loading','parent_calibration_energy']))
basefit=fit[fit.variant=='base']
fitmetrics=['train_target_mse','validation_target_mse','truth_test_mse','aggregate_truth_test_mse']
csv('fit_summary.csv',summary(basefit,['task','optimizer','steps','topology'],fitmetrics))
csv('fit_Adam_GD_paired.csv',paired(basefit,['task','steps'],fitmetrics))
csv('superposition_summary.csv',summary(sup,['task','optimizer','steps','amplitude','test'],['relative_rmse']))
if AMPLITUDE:
    amp=pd.DataFrame(AMPLITUDE)
    csv('amplitude_summary.csv',summary(amp,['task','optimizer','steps','amplitude','probe','depth'],['actual_response_rms','response_rms_per_delta','homogeneity_rmse','homogeneity_relative_rmse']))
# Audit the original table at its reported three-decimal precision.
audit=[]
for task,reference in [('linear',.420),('nonlinear',.714)]:
    a=primary[(primary.optimizer=='adam')&(primary.task==task)]
    value=float(a.mean_node_relative_rmse.mean())
    audit.append(dict(task=task,measured=value,paper_rounded_reference=reference,rounds_to_paper_value=bool(round(value,3)==reference),scope='full_original' if MODE=='full' else 'smoke_not_paper_comparable'))
csv('paper_Table1_audit.csv',audit)

# Read existing root training histories without modifying them.
curves=[]
for seed in SEEDS:
    for task in TASKS:
        for opt in OPTIMIZERS:
            for steps in BUDGETS:
                z=graph(seed,task,'chain',opt,steps,'base');mid=str(z['model_ids'][0]);path=RUN/'model_cache'/mid/f'completed_{steps}.npz'
                model=None
                if path.is_file():model=read_npz(path)
                elif SOURCE.exists(f'model_cache/{mid}/completed.npz'):model=SOURCE.npz(f'model_cache/{mid}/completed.npz')
                if model is None or 'history' not in model:continue
                for h in np.asarray(model['history']):
                    curves.append(dict(seed=seed,task=task,optimizer=opt,graph_budget=steps,update=int(h[0]),train_mse=float(h[1]),gradient_norm=float(h[2]) if len(h)>2 else np.nan))
if curves:csv('root_training_curves.csv',curves)

def savefig(name):
    plt.tight_layout();plt.savefig(REPORT/name,dpi=160,bbox_inches='tight');plt.close()
colors={'adam':'#B95040','gd':'#287C9A'}
fig,axs=plt.subplots(1,2,figsize=(10,3.8))
for ax,task in zip(axs,TASKS):
    q=primary[primary.task==task]
    for j,opt in enumerate(OPTIMIZERS):
        g=q[q.optimizer==opt];values=[g.mean_node_relative_rmse.mean(),g.pooled_relative_rmse.mean()]
        ax.bar(np.arange(2)+(j-.5)*.32,values,width=.32,label=opt.upper(),color=colors[opt])
    ax.set_xticks([0,1],['Mean node rRMSE','Pooled rRMSE']);ax.set_title(task);ax.legend()
savefig('01_original_vs_energy_metric.png')
fig,axs=plt.subplots(2,3,figsize=(13,7))
for row,task in enumerate(TASKS):
    for opt in OPTIMIZERS:
        q=d[(d.task==task)&(d.topology=='chain')&(d.optimizer==opt)&(d.steps==BASE_STEPS)&(d.amplitude==1.)&(d.method=='calibrated_recursive')]
        for col,metric in enumerate(['response_rms','rmse','relative_rmse']):
            a=q.groupby(['seed','depth'])[metric].mean().groupby('depth').mean()
            axs[row,col].plot(a.index,a.values,'o-',label=opt.upper(),color=colors[opt]);axs[row,col].set_title(task+' | '+metric);axs[row,col].set_xlabel('Depth');axs[row,col].legend()
            if col<2:axs[row,col].set_yscale('log')
savefig('02_chain_depth.png')
fig,axs=plt.subplots(2,2,figsize=(10,7))
for row,task in enumerate(TASKS):
    for opt in OPTIMIZERS:
        q=seed_metrics[(seed_metrics.task==task)&(seed_metrics.optimizer==opt)&(seed_metrics.amplitude==1.)&(seed_metrics.method=='calibrated_recursive')]
        a=q.groupby('steps').mean_node_relative_rmse.mean()
        axs[row,0].plot(a.index,a.values,'o-',label=opt.upper(),color=colors[opt]);axs[row,0].set_title(task+' | Response rRMSE')
        g=basefit[(basefit.task==task)&(basefit.optimizer==opt)].groupby(['seed','steps']).truth_test_mse.mean().groupby('steps').mean()
        axs[row,1].plot(g.index,g.values,'o-',label=opt.upper(),color=colors[opt]);axs[row,1].set_title(task+' | Truth test MSE')
    for ax in axs[row]:ax.set_xlabel('Updates');ax.legend()
savefig('03_budget_and_task_fit.png')
if AMPLITUDE:
    fig,axs=plt.subplots(1,2,figsize=(10,4))
    for ax,task in zip(axs,TASKS):
        for opt in OPTIMIZERS:
            q=d[(d.task==task)&(d.topology=='chain')&(d.optimizer==opt)&(d.steps==BASE_STEPS)&(d.method=='calibrated_recursive')]
            for depth in sorted(q.depth.unique()):
                a=q[q.depth==depth].groupby(['seed','amplitude']).response_rms.mean().groupby('amplitude').mean().sort_index()
                ax.plot(a.index*CFG['delta'],a.values,'o-',alpha=.7,color=colors[opt],label=f'{opt} d={depth}')
        ax.set_xscale('log');ax.set_yscale('log');ax.set_title(task);ax.set_xlabel('Root-target perturbation scale');ax.set_ylabel('Measured response RMS');ax.legend(fontsize=6,ncol=2)
    savefig('04_amplitude_response.png')

lines=['# Original-network optimizer comparison',
       'Fixed width16, no L2, paired data and initialization, full-batch training, learning rate 0.01.',
       'The primary table uses 600 updates in full mode. Percentage reductions use the ratio of seed means.',
       'Complete response, task-fit, budget, amplitude, and audit tables are saved alongside this report.']
for task in TASKS:
    row=primary_compare[(primary_compare.task==task)&(primary_compare.metric=='mean_node_relative_rmse')].iloc[0]
    lines.append(f"{task}: Adam={row.adam_mean:.6f}; GD={row.gd_mean:.6f}; reduction={row.reduction_pct_of_means:.2f}%; GD lower in {row.gd_lower_seeds}/{row.n_pairs} seeds.")
atomic_text(REPORT/'README_results.md','\n\n'.join(lines))
atomic_text(REPORT/'COMPLETE.json',json.dumps(dict(mode=MODE,seeds=SEEDS,budgets=BUDGETS,amplitudes=AMPLITUDES,counters=COUNTS,elapsed_minutes=(time.time()-START)/60),indent=2))
archive=RUN/'optimizer_check_tables_figures.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(REPORT.iterdir()):
        if p.is_file():z.write(p,p.name)
    z.write(RUN/'training_plan.json','training_plan.json')
print('Optimizer tables and figures saved.')
display(primary_compare[primary_compare.metric=='mean_node_relative_rmse'])
if SHOW_ALL_TABLES:
    for name in ['response_summary.csv','fit_Adam_GD_paired.csv','budget_Adam_GD_paired.csv']:
        print(name);display(pd.read_csv(REPORT/name))
OPTIMIZER_REPORT=REPORT
OPTIMIZER_RUN=RUN
OPTIMIZER_PRIMARY=primary_compare.copy()
if SOURCE.z:SOURCE.z.close()


## I. Supplement: propagation, population growth, and teacher choice

The four width-128, 600-update settings are plain tanh and linear-skip-plus-tanh,
each with L2 coefficient 0 or 0.01. All four settings are computed across the three
experiments. The inline presentation highlights the regularized settings; complete
results are saved for every setting. The original width-16 network and ridge remain
reference learners. Structural/L2 interaction contrasts are not added here.

| ID | Student architecture | L2 | Width | Updates |
|---|---|---:|---:|---:|
| plain_0 | tanh | 0 | 128 | 600 |
| plain_l2 | tanh | 0.01 | 128 | 600 |
| residual_0 | linear skip + tanh | 0 | 128 | 600 |
| residual_l2 | linear skip + tanh | 0.01 | 128 | 600 |

The original two-teacher/four-student grid is retained. Its scale extension uses
three teachers and eight students, gaps 0.1 and 0.35, and overlaps 0, 0.5, and 0.9.
Both tasks, all four new networks, and ridge are included. The original width-16
teacher-choice baseline is available for the linear two-teacher experiment.


In [ ]:
REPORT_KIND='anonymous_english'


In [ ]:
import os,sys,json,time,hashlib,io,zipfile,platform,itertools,importlib.util,subprocess
from pathlib import Path
for k in ['OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS']:os.environ.setdefault(k,'1')
for pkg in ['numpy','pandas','matplotlib','threadpoolctl']:
    if importlib.util.find_spec(pkg) is None:subprocess.check_call([sys.executable,'-m','pip','install',pkg])
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
THREAD_LIMIT=threadpool_limits(limits=1)
try:from IPython.display import display,Markdown
except ImportError:
    display=print;Markdown=str
try:
    import google.colab
    IN_COLAB=True
except ImportError:IN_COLAB=False
if IN_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive')
SOURCE_DIR=Path(os.environ.get('LINEAGE_SOURCE','/content/drive/MyDrive/lineage_dag_full/b23c247b3a094046'))
SOURCE_ZIP=Path(os.environ.get('LINEAGE_SOURCE_ZIP',str(SOURCE_DIR)+'_full_results.zip'))
PREVIOUS_NETWORK=Path(os.environ.get('LINEAGE_PREVIOUS_NETWORK','/content/drive/MyDrive/lineage_network_check/network_capacity_budget_v1_full_62def0c0ce6e'))
PREVIOUS_L2=Path(os.environ.get('LINEAGE_PREVIOUS_L2','/content/drive/MyDrive/lineage_network_l2/network_l2_v1_full_ba579e266c93'))
PREVIOUS_RESIDUAL=Path(os.environ.get('LINEAGE_PREVIOUS_RESIDUAL','/content/drive/MyDrive/lineage_network_residual/network_residual_v1_full_8a6980fa2fee'))
OUTPUT_ROOT=Path(os.environ.get('LINEAGE_INTEGRATED_OUTPUT','/content/drive/MyDrive/lineage_rebuttal_integrated' if IN_COLAB else './lineage_rebuttal_integrated'))
MODE=os.environ.get('LINEAGE_INTEGRATED_MODE','full')
RUN_EXTENSION=os.environ.get('LINEAGE_RUN_EXTENSION','1')=='1' # Preserve the original experiment configuration.
MECHANISM_AUTO_DOWNLOAD=False
TRAIN_SHA='f7c132a5c62c1251b12f3ec765c6186e251ea9d7a3b2693aade34bd679f8ceb2'
ORIGINAL_SHA='1ac6b1a53dbbb84842cf1ff69735acec25783f612a8b1cc61a82696fc900d01d'
VERSION='integrated_rebuttal_v1'
START=time.time();COUNTS=dict(new_models=0,old_models=0,cached_models=0,resumed_models=0,old_graphs=0,cached_graphs=0,new_graphs=0,updates=0)
EPS=1e-16

def atomic_text(path,s):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);q=path.with_name(path.name+'.tmp');q.write_text(s,encoding='utf-8');os.replace(q,path)
def atomic_npz(path,**obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);q=path.with_name(path.name+'.tmp')
    with open(q,'wb') as f:np.savez_compressed(f,**obj)
    os.replace(q,path)
def read_npz(path):
    with np.load(path,allow_pickle=False) as z:return {k:z[k].copy() for k in z.files}
def digest(o):return hashlib.sha256(json.dumps(o,sort_keys=True,separators=(',',':')).encode()).hexdigest()
def csv(name,rows):
    d=pd.DataFrame(rows);atomic_text(REPORT/name,d.to_csv(index=False));return d
def log(s):print(f'{s} | {(time.time()-START)/60:.1f} min | {COUNTS}',flush=True)


### I1. Protocol, data, and existing caches
Read exact original arrays and split indices. Verify previous cache metadata. Reporting settings are excluded from training-cache identities.

In [ ]:
class Source:
    def __init__(self,directory,archive):
        self.directory=directory;self.z=None;self.prefix='';self.names=set();self.accessed={}
        if archive.is_file():
            self.z=zipfile.ZipFile(archive);self.names=set(self.z.namelist())
            pp=[n for n in self.names if n=='protocol.json' or n.endswith('/protocol.json')]
            if len(pp)!=1:raise ValueError('Expected exactly one protocol.json in the source archive')
            self.prefix=pp[0][:-len('protocol.json')]
        if not self.exists('protocol.json'):raise FileNotFoundError(f'Source data not found: {directory} or {archive}')
        if self.z and (directory/'protocol.json').is_file():
            assert json.loads((directory/'protocol.json').read_text())==json.loads(self.z.read(self.prefix+'protocol.json'))
    def exists(self,rel):return (self.directory/rel).is_file() or self.prefix+rel in self.names
    def read(self,rel):
        p=self.directory/rel
        if p.is_file():b=p.read_bytes()
        elif self.z and self.prefix+rel in self.names:b=self.z.read(self.prefix+rel)
        else:raise FileNotFoundError(rel)
        self.accessed[rel]=hashlib.sha256(b).hexdigest();return b
    def npz(self,rel):
        with np.load(io.BytesIO(self.read(rel)),allow_pickle=False) as z:return {k:z[k].copy() for k in z.files}
SOURCE=Source(SOURCE_DIR,SOURCE_ZIP);PROTOCOL=json.loads(SOURCE.read('protocol.json'));CFG=PROTOCOL['config']
assert PROTOCOL['code_sha']==ORIGINAL_SHA
for k,v in dict(mode='full',d=8,train=192,validation=256,test=1024,nodes=16,prop_nodes=8,hidden=16,beta=.7,delta=.15,learning_rate=.01,ridge=.05).items():
    assert CFG[k]==v,f'Source protocol mismatch: {k}={CFG[k]}'
assert CFG['seeds']==list(range(2701,2709))
if MODE=='full':
    SEEDS=CFG['seeds'];WIDTH=128;BUDGET=600;PROP_M=8;GROW_M=16;SIZES=[4,8,16];SLOTS=4;EXT_SLOTS=8
    GAPS=list(CFG['quality_gaps']);RHOS=list(CFG['rhos']);EXT_GAPS = list(CFG['quality_gaps']);EXT_RHOS=[0.,.5,.9]
    BOOT=4000;SAVE_EVERY=100;LOG_EVERY=50
elif MODE=='smoke':
    SEEDS=CFG['seeds'][:2];WIDTH=12;BUDGET=8;PROP_M=3;GROW_M=4;SIZES=[2,3,4];SLOTS=2;EXT_SLOTS=3
    GAPS=[0.,.2];RHOS=[0.,.9];EXT_GAPS=[.2];EXT_RHOS=[.5];BOOT=100;SAVE_EVERY=4;LOG_EVERY=4
else:raise ValueError('MODE must be full or smoke')
TASKS=['linear','nonlinear'];TOPOS=['roots','star','chain','multi_parent'];BETAS=[.4,.7]
VARIANTS={'base':(0,0),'h1':(1,0),'h2':(0,1),'negative':(-1,0),'sum':(1,1)}
N=CFG['train'];VEND=N+CFG['validation'];TOTAL=VEND+CFG['test'];TR=slice(0,N);VA=slice(N,VEND);TE=slice(VEND,None)
DATA={}
for seed in SEEDS:
    z=SOURCE.npz(f'data/{seed}.npz');X,theta,noise,dirs=[z[k] for k in ['X','theta','noise','dirs']]
    assert X.shape==(TOTAL,8) and noise.shape==(16,8) and dirs.shape==(2,8)
    for k,e in [('train_indices',np.arange(N)),('validation_indices',np.arange(N,VEND)),('test_indices',np.arange(VEND,TOTAL))]:np.testing.assert_array_equal(z[k],e)
    np.testing.assert_allclose(X[TR].T@X[TR]/N,np.eye(8),atol=1e-10);assert np.isfinite(X).all()
    DATA[seed]=(X,theta,noise,dirs)
DATA_HASH=dict(SOURCE.accessed)
MAIN_IDS=['plain_0','plain_l2','residual_0','residual_l2']
SETTINGS={k:dict(id=k,kind='mlp',architecture=a,width=WIDTH,l2=l,budget=BUDGET) for k,a,l in [('plain_0','plain',0.),('plain_l2','plain',.01),('residual_0','residual',0.),('residual_l2','residual',.01)]}
SETTINGS.update(paper_nn=dict(id='paper_nn',kind='mlp',architecture='plain',width=16,l2=0.,budget=600 if MODE=='full' else BUDGET),ridge_005=dict(id='ridge_005',kind='ridge',architecture='linear',width=0,l2=.05,budget=0),ridge_0=dict(id='ridge_0',kind='ridge',architecture='linear',width=0,l2=0.,budget=0))
SETTINGS['paper_nn_200']=dict(SETTINGS['paper_nn'],id='paper_nn_200',budget=200 if MODE=='full' else max(4,BUDGET//2))
OLD_RUNS=[]
for tag,path,sha in [('capacity',PREVIOUS_NETWORK,'3557dfd1e28aaf3013a0b28b89e22357a957a66ffbe143592d68f6eb1838956b'),('l2',PREVIOUS_L2,'9b9eb48df12084231b816b4731b03bf30322d93a8562d6c6e2e591c154ef05ba'),('residual',PREVIOUS_RESIDUAL,'986542867a5ea1f380aba71eab2e478101d95e803f4ee79eec378c6f261d1bb9')]:
    if MODE=='full' and (path/'plan.json').is_file():
        p=json.loads((path/'plan.json').read_text());assert p['build_sha']==sha,f'{tag} code version mismatch'
        assert p['lr']==CFG['learning_rate'] and p['dtype']=='float64'
        for k,v in DATA_HASH.items():assert p['source_hashes'][k]==v,f'{tag} data mismatch: {k}'
        OLD_RUNS.append(dict(tag=tag,path=path,sha=sha))
# Reporting settings do not participate in the training-cache key.
CACHE_PLAN=dict(version=VERSION,train_sha=TRAIN_SHA,mode=MODE,source_hashes=DATA_HASH,settings=SETTINGS,lr=CFG['learning_rate'],beta=CFG['beta'],delta=CFG['delta'],numpy=np.__version__,python=platform.python_version())
RUN=OUTPUT_ROOT/(VERSION+'_'+MODE+'_'+digest(CACHE_PLAN)[:12]);REPORT=RUN/'reports'/REPORT_KIND
assert SOURCE_DIR.resolve() not in RUN.resolve().parents and SOURCE_DIR.resolve()!=RUN.resolve()
REPORT.mkdir(parents=True,exist_ok=True)
atomic_text(RUN/'training_plan.json',json.dumps(CACHE_PLAN,indent=2));atomic_text(REPORT/'experiment_plan.json',json.dumps(dict(mode=MODE,report_kind=REPORT_KIND,main_settings=MAIN_IDS,reference_settings=['paper_nn','ridge_005','ridge_0'],tasks=TASKS,seeds=SEEDS,propagation_nodes=PROP_M,growth_sizes=SIZES,betas=BETAS,gaps=GAPS,rhos=RHOS,choice_students=SLOTS,extension=RUN_EXTENSION,extension_teachers=3,extension_students=EXT_SLOTS,extension_gaps=EXT_GAPS,extension_rhos=EXT_RHOS,source_dir=str(SOURCE_DIR),old_runs=[dict(tag=p['tag'],path=str(p['path']),sha=p['sha']) for p in OLD_RUNS]),indent=2))
if (REPORT/'COMPLETE.json').exists():(REPORT/'COMPLETE.json').unlink()
print('Mode: ',MODE,'Report: ',REPORT_KIND,'\nShared training directory: ',RUN,'\nReport directory: ',REPORT)
print('Verified previous caches: ',[r['tag'] for r in OLD_RUNS]);print('Three-teacher extension: ',RUN_EXTENSION)


### I2. Learners and graph training

$f_{\mathrm{plain}}(x)=v^\top\tanh(U^\top x+b)+c$;
$f_{\mathrm{skip}}(x)=a^\top x+f_{\mathrm{plain}}(x)$.
The skip vector starts at zero; other parameters share initialization.
The neural objective is
$L=\operatorname{MSE}+\lambda(\|U\|_F^2/h+\|v\|^2+\|a\|^2)$,
with the last term present only for the skip architecture. Biases are unpenalized.
All parameters use full-batch Adam for 600 updates at learning rate 0.01.
Ridge retains the original closed-form solution. Growth reads 4/8/16-node prefixes
of the same 16-node trained graph.

In [ ]:
def truth_fn(X,theta,task):return X@theta+(0.3*np.sin(X[:,0]*X[:,1]) if task=='nonlinear' else 0.)
def parents(topo,m):
    W=np.zeros((m,m))
    if topo=='star':W[1:,0]=1
    elif topo=='chain':W[np.arange(1,m),np.arange(m-1)]=1
    elif topo=='multi_parent':
        for i in range(1,m):
            js=sorted(set([0,i-1]));W[i,js]=1/len(js)
    elif topo!='roots':raise ValueError(topo)
    return W

def init_net(seed,s):
    h=s['width'];r=np.random.default_rng(seed);d=CFG['d']
    p=[r.normal(size=(d,h))/np.sqrt(d),np.zeros(h),r.normal(size=h)/np.sqrt(h),np.zeros(1)]
    if s['architecture']=='residual':p.append(np.zeros(d))
    return p

def forward(X,p):
    H=np.tanh(X@p[0]+p[1]);y=H@p[2]+p[3][0]
    if len(p)==5:y=y+X@p[4]
    return y,H

def objective_grad(X,y,p,lam):
    pred,H=forward(X,p);err=pred-y;g=2*err/len(y);dh=(g[:,None]*p[2])*(1-H*H)
    grad=[X.T@dh,dh.sum(0),H.T@g,np.array([g.sum()])]
    pen=np.sum(p[0]**2)/p[0].shape[1]+np.sum(p[2]**2)
    if len(p)==5:grad.append(X.T@g);pen+=np.sum(p[4]**2)
    if lam:
        grad[0]+=2*lam*p[0]/p[0].shape[1];grad[2]+=2*lam*p[2]
        if len(p)==5:grad[4]+=2*lam*p[4]
    return float(np.mean(err**2)+lam*pen),grad

def model_signature(X,target,seed,s):
    # Include all targets to preserve validation-history cache identity.
    meta=dict(train_sha=TRAIN_SHA,init_seed=int(seed),kind=s['kind'],architecture=s['architecture'],width=s['width'],l2=s['l2'],budget=s['budget'],lr=CFG['learning_rate'],dtype='float64')
    return hashlib.sha256(X.tobytes()+target.tobytes()+json.dumps(meta,sort_keys=True).encode()).hexdigest()

def old_model(X,target,seed,s):
    if MODE!='full':return None
    if s['kind']=='ridge' or (s['architecture']=='plain' and s['width']==16 and s['l2']==0):
        kind='ridge' if s['kind']=='ridge' else 'mlp';budget=0 if kind=='ridge' else s['budget']
        for lam in ([s['l2']] if kind=='ridge' else [.05,0.]):
            sig=hashlib.sha256(X.tobytes()+target[TR].tobytes()+str((int(seed),kind,budget,lam,ORIGINAL_SHA)).encode()).hexdigest()
            rel=f'model_cache/{sig}/completed.npz'
            if SOURCE.exists(rel):
                z=SOURCE.npz(rel);assert str(z['signature'])==sig
                return z,'original_model:'+sig
    if s['kind']=='mlp':
        for r in reversed(OLD_RUNS):
            if r['tag']=='capacity' and (s['architecture']!='plain' or s['l2']!=0):continue
            if r['tag']=='l2' and s['architecture']!='plain':continue
            meta=dict(init_seed=int(seed),width=int(s['width']),lr=CFG['learning_rate'],build_sha=r['sha'],dtype='float64')
            if r['tag'] in ['l2','residual']:meta['l2']=float(s['l2'])
            if r['tag']=='residual':meta['architecture']=s['architecture']
            sig=hashlib.sha256(X.tobytes()+target.tobytes()+json.dumps(meta,sort_keys=True).encode()).hexdigest()
            p=r['path']/'model_cache'/sig/f"completed_{s['budget']}.npz"
            if p.is_file():
                z=read_npz(p);assert str(z['signature'])==sig
                return z,r['tag']+':'+sig
    return None

def fit_model(X,target,seed,s,interrupt_step=None,allow_old=True):
    sig=model_signature(X,target,seed,s);folder=RUN/'model_cache'/sig;done=folder/'completed.npz'
    if done.is_file():
        z=read_npz(done);assert str(z['signature'])==sig;COUNTS['cached_models']+=1;return z,sig
    found=old_model(X,target,seed,s) if allow_old else None
    if found:
        z,origin=found;assert z['pred'].shape==(TOTAL,) and np.isfinite(z['pred']).all()
        COUNTS['old_models']+=1;return z,origin
    folder.mkdir(parents=True,exist_ok=True)
    if s['kind']=='ridge':
        coef=np.linalg.solve(X[TR].T@X[TR]/N+s['l2']*np.eye(CFG['d']),X[TR].T@target[TR]/N)
        pred=X@coef;z=dict(signature=sig,pred=pred,train_mse=float(np.mean((pred[TR]-target[TR])**2)),coef=coef)
        atomic_npz(done,**z);COUNTS['new_models']+=1;return z,sig
    budget=s['budget'];ck=folder/'checkpoint.npz';h=[]
    if ck.is_file():
        z=read_npz(ck);assert str(z['signature'])==sig;start=int(z['step']);assert start<=budget
        count=5 if s['architecture']=='residual' else 4
        p=[z[f'p{k}'] for k in range(count)];m=[z[f'm{k}'] for k in range(count)];v=[z[f'v{k}'] for k in range(count)];h=z['history'].tolist();COUNTS['resumed_models']+=1
    else:
        p=init_net(seed,s);m=[np.zeros_like(a) for a in p];v=[np.zeros_like(a) for a in p];start=0
    def pack(step):
        z=dict(signature=sig,step=step,history=np.asarray(h,float),setting_json=json.dumps(s))
        for k in range(len(p)):z.update({f'p{k}':p[k],f'm{k}':m[k],f'v{k}':v[k]})
        return z
    for step in range(start+1,budget+1):
        loss,gg=objective_grad(X[TR],target[TR],p,s['l2'])
        if not np.isfinite(loss) or not all(np.isfinite(g).all() for g in gg):raise FloatingPointError(sig)
        for k in range(len(p)):
            m[k]=.9*m[k]+.1*gg[k];v[k]=.999*v[k]+.001*gg[k]**2
            p[k]-=CFG['learning_rate']*(m[k]/(1-.9**step))/(np.sqrt(v[k]/(1-.999**step))+1e-8)
        COUNTS['updates']+=1
        if step%LOG_EVERY==0 or step==budget or step==interrupt_step:
            tr=float(np.mean((forward(X[TR],p)[0]-target[TR])**2));va=float(np.mean((forward(X[VA],p)[0]-target[VA])**2))
            h.append([step,tr,va])
        if step%SAVE_EVERY==0 or step==budget or step==interrupt_step:atomic_npz(ck,**pack(step))
        if step==interrupt_step:raise InterruptedError('Test interruption; checkpoint saved')
    z=pack(budget);pred=forward(X,p)[0];loss,gg=objective_grad(X[TR],target[TR],p,s['l2'])
    assert np.isfinite(pred).all()
    z.update(pred=pred,train_mse=float(np.mean((pred[TR]-target[TR])**2)),objective=loss,gradient_norm=float(np.sqrt(sum(np.sum(g*g) for g in gg))))
    atomic_npz(done,**z);COUNTS['new_models']+=1;return z,sig

def local_target(X,truth,noise,i,dirs=None,variant='base'):
    y=np.empty(len(X));a,b=VARIANTS[variant]
    for sl in [TR,VA,TE]:
        y[sl]=truth[sl]+X[sl]@noise[i]
        if i==0 and dirs is not None:y[sl]+=CFG['delta']*X[sl]@(a*dirs[0]+b*dirs[1])
    return y

def target_for(X,truth,noise,dirs,W,P,i,variant,beta):
    y=local_target(X,truth,noise,i,dirs,variant)
    return y if not W[i].sum() else (1-beta)*y+beta*(W[i,:i]@np.asarray(P))

def old_graph(seed,task,topo,m,beta,variant,s):
    if MODE!='full':return None
    candidates=[]
    if s['kind']=='ridge' or (s['architecture']=='plain' and s['width']==16 and s['l2']==0):
        kind='ridge' if s['kind']=='ridge' else 'mlp'
        for lam in ([s['l2']] if kind=='ridge' else [.05,0.]):
            case=dict(seed=int(seed),learner=kind,topology=topo,M=int(m),budget=(int(s['budget']) if kind=='mlp' else 600),beta=float(beta),lam=float(lam),task=task,variant=variant)
            key=hashlib.sha256(json.dumps(case,sort_keys=True).encode()).hexdigest()[:20];rel=f'graph_predictions/{key}.npz'
            if SOURCE.exists(rel):candidates.append((SOURCE.npz(rel),case,'original_graph'))
    if s['kind']=='mlp' and beta==.7 and m==8:
        for r in reversed(OLD_RUNS):
            if r['tag']=='capacity' and (s['architecture']!='plain' or s['l2']!=0):continue
            if r['tag']=='l2' and s['architecture']!='plain':continue
            case=dict(seed=int(seed),task=task,topology=topo,variant=variant,width=int(s['width']),budget=int(s['budget']),M=int(m))
            if r['tag'] in ['l2','residual']:case['l2']=float(s['l2'])
            if r['tag']=='residual':case['architecture']=s['architecture']
            key=hashlib.sha256(json.dumps(case,sort_keys=True).encode()).hexdigest()[:20];path=r['path']/'graph_predictions'/f'{key}.npz'
            if path.is_file():candidates.append((read_npz(path),case,r['tag']+'_graph'))
    for z,case,origin in candidates:
        assert json.loads(str(z['case_json']))==case
        assert z['pred'].shape==(m,TOTAL) and np.isfinite(z['pred']).all()
        np.testing.assert_array_equal(z['W'],parents(topo,m))
        np.testing.assert_allclose(z['truth'],truth_fn(DATA[seed][0],DATA[seed][1],task),rtol=1e-12,atol=1e-12)
        return z,origin
    return None

def graph(seed,task,topo,m,beta,variant,s):
    case=dict(seed=int(seed),task=task,topology=topo,M=int(m),beta=float(beta),variant=variant,setting=s['id'])
    path=RUN/'graph_predictions'/(digest(case)[:24]+'.npz')
    if path.is_file():
        z=read_npz(path);assert json.loads(str(z['case_json']))==case;COUNTS['cached_graphs']+=1;return z
    found=old_graph(seed,task,topo,m,beta,variant,s)
    if found:
        z,origin=found;z['case_json']=json.dumps(case);z['origin']=origin
        atomic_npz(path,**z);COUNTS['old_graphs']+=1;return z
    X,theta,noise,dirs=DATA[seed];truth=truth_fn(X,theta,task);W=parents(topo,m);P=[];ids=[];losses=[]
    if m>PROP_M:
        pre=graph(seed,task,topo,PROP_M,beta,variant,s)
        P=list(pre['pred']);ids=list(pre['model_ids']);losses=list(pre['train_mse'])
    for i in range(len(P),m):
        target=target_for(X,truth,noise,dirs,W,P,i,variant,beta)
        z,sig=fit_model(X,target,seed*100+i,s);P.append(z['pred']);ids.append(sig);losses.append(z['train_mse'])
    z=dict(pred=np.asarray(P),truth=truth,W=W,model_ids=np.asarray(ids),train_mse=np.asarray(losses),case_json=json.dumps(case),origin='new')
    atomic_npz(path,**z);COUNTS['new_graphs']+=1
    return z


### I3. Propagation and growth metrics
Calibrate on h1 validation responses and evaluate held-out directions. Save superposition and root controls. Growth records aggregate and individual MSE, bias, covariance, and effective rank.

In [ ]:
PROP=[];SUPER=[];RISK=[];GROW=[];CHOICE=[];MARGINS=[];BALANCED=[];FIT=[];RIDGE=[];CONTROL=[];LOADINGS=[];DONE=[]

def meta(s):return dict(setting=s['id'],architecture=s['architecture'],l2=s['l2'],width=s['width'],budget=s['budget'])
def boot(a):
    a=np.asarray(a,float);a=a[np.isfinite(a)]
    if not len(a):return np.nan,np.nan,np.nan
    if len(a)==1:return float(a[0]),np.nan,np.nan
    ix=np.random.default_rng(20261108).integers(0,len(a),(BOOT,len(a)));b=a[ix].mean(1)
    return float(a.mean()),float(np.quantile(b,.025)),float(np.quantile(b,.975))
def summarize(df,keys,metrics):
    rows=[]
    if df.empty:return pd.DataFrame()
    for key,g in df.groupby(keys,dropna=False):
        if not isinstance(key,tuple):key=(key,)
        for metric in metrics:
            a=g.groupby('seed')[metric].mean();mean,lo,hi=boot(a)
            rows.append(dict(zip(keys,key),metric=metric,mean=mean,ci_low=lo,ci_high=hi,n_seeds=int(a.notna().sum()),positive_seeds=int((a>0).sum()),negative_seeds=int((a<0).sum()),finite_rows=int(np.isfinite(g[metric]).sum()),rows=len(g)))
    return pd.DataFrame(rows)
def metrics(P,y):
    E=P-y;bar=E.mean(0);C=np.atleast_2d(np.cov(E,bias=True));m=len(E);ev=np.linalg.eigvalsh(C)
    return dict(aggregate_mse=float(np.mean(bar**2)),variance=float(np.var(bar)),bias_squared=float(np.mean(bar)**2),individual_mse=float(np.mean(E**2)),offdiag_cov=float((C.sum()-np.trace(C))/(m*(m-1))) if m>1 else np.nan,effective_rank=float(np.trace(C)/ev[-1]) if ev[-1]>EPS else np.nan)
def score(obs,hat):
    mse=float(np.mean((obs-hat)**2));energy=float(np.mean(obs**2));ratio=mse/energy if energy>EPS else np.nan
    return dict(mse=mse,response_energy=energy,relative_rmse=float(np.sqrt(ratio)),skill=1-ratio)
def fit_A(D,W):
    A=np.zeros_like(W)
    for i in range(1,len(W)):
        if not W[i].sum():continue
        x=W[i]@D[:,VA];den=float(x@x);A[i]=float(x@D[i,VA])/den*W[i] if den>EPS else np.nan
    return A

def response_prediction(obs,A,W,recursive):
    H=np.zeros_like(obs);H[0]=obs[0]
    for i in range(1,len(W)):
        js=np.flatnonzero(W[i])
        if len(js):H[i]=A[i,js]@(H[js] if recursive else obs[js])
    return H

def record_fit(seed,task,s,stage,case,P,truth,targets):
    for i,target in enumerate(targets):
        FIT.append(dict(seed=seed,task=task,stage=stage,case=case,node=i,**meta(s),train_mse=float(np.mean((P[i,TR]-target[TR])**2)),validation_mse=float(np.mean((P[i,VA]-target[VA])**2)),truth_test_mse=float(np.mean((P[i,TE]-truth[TE])**2))))

def evaluate_propagation(seed,task,topo,s,bank):
    P=bank['base']['pred'];y=bank['base']['truth'];W=bank['base']['W'];D={k:z['pred']-P for k,z in bank.items() if k!='base'};A=fit_A(D['h1'],W)
    mm=dict(seed=seed,task=task,topology=topo,**meta(s));depth=np.zeros(len(W),int)
    for i in range(1,len(W)):
        js=np.flatnonzero(W[i]);depth[i]=1+max(depth[js]) if len(js) else 0
        if len(js):LOADINGS.append(dict(**mm,node=i,loading=float(A[i].sum()),identified=bool(np.isfinite(A[i]).all())))
    E=P[:,TE]-y[TE];L0=float(np.mean(E.mean(0)**2))
    for probe in ['h2','negative']:
        obs=D[probe][:,TE];nom=(CFG['beta']/(1+s['l2']) if s['kind']=='ridge' else CFG['beta'])*W
        zero=np.zeros_like(obs);zero[0]=obs[0]
        hats={'one_step':response_prediction(obs,A,W,False),'recursive':response_prediction(obs,A,W,True),'nominal_recursive':response_prediction(obs,nom,W,True),'no_transmission':zero}
        if topo=='roots':CONTROL.append(dict(**mm,probe=probe,descendant_response_rms=float(np.sqrt(np.mean(obs[1:]**2)))))
        for method,H in hats.items():
            # Keep zero-response root controls separate from ratio summaries.
            if topo!='roots':
                for i in range(1,len(W)):PROP.append(dict(**mm,probe=probe,method=method,node=i,depth=int(depth[i]),**score(obs[i],H[i])))
            actual=float(np.mean((E.mean(0)+obs.mean(0))**2));pred=float(np.mean((E.mean(0)+H.mean(0))**2))
            RISK.append(dict(**mm,probe=probe,method=method,baseline_mse=L0,actual_change=actual-L0,predicted_change=pred-L0,absolute_error=abs(pred-actual),absolute_error_pct_baseline=100*abs(pred-actual)/L0 if L0>EPS else np.nan))
    for name,res,ref in [('superposition',D['sum']-D['h1']-D['h2'],D['h1']+D['h2']),('sign',D['negative']+D['h1'],D['h1'])]:
        num=float(np.mean(res[:,TE]**2));den=float(np.mean(ref[:,TE]**2))
        SUPER.append(dict(**mm,test=name,mse=num,reference_energy=den,relative_rmse=np.sqrt(num/den) if den>EPS else np.nan))
    X,theta,noise,dirs=DATA[seed]
    targets=[target_for(X,y,noise,dirs,W,P[:i],i,'base',CFG['beta']) for i in range(len(W))]
    record_fit(seed,task,s,'propagation',topo,P,y,targets)
    if s['kind']=='ridge' and task=='linear':
        c=np.where(W.sum(1)>0,1-CFG['beta'],1);U=X[TE]@((c[:,None]*noise[:len(W)]-s['l2']*theta)/(1+s['l2'])).T
        Phi=np.linalg.solve(np.eye(len(W))-CFG['beta']/(1+s['l2'])*W,np.eye(len(W)))
        err=float(np.max(abs(E-Phi@U.T)));assert err<1e-8
        RIDGE.append(dict(**mm,stage='propagation',max_identity_error=err))

def evaluate_growth(seed,task,topo,beta,s,z):
    P=z['pred'];y=z['truth'];W=z['W'];X,theta,noise,dirs=DATA[seed]
    for size in SIZES:
        r=metrics(P[:size,TE],y[TE]);ev=P[:size,VA]-y[VA];scale=ev.std(1)
        standard=(P[:size,TE]-y[TE]-ev.mean(1)[:,None])/np.maximum(scale[:,None],1e-12)
        row=dict(seed=seed,task=task,topology=topo,beta=beta,M=size,**meta(s),**r,standardized_aggregate_variance=float(np.var(standard.mean(0))) if np.all(scale>1e-12) else np.nan)
        if s['kind']=='ridge' and task=='linear':
            Phi=np.linalg.solve(np.eye(size)-beta/(1+s['l2'])*W[:size,:size],np.eye(size));v=Phi.sum(0)/size;c=np.where(W[:size,:size].sum(1)>0,1-beta,1)
            U=X[TE]@((c[:,None]*noise[:size]-s['l2']*theta)/(1+s['l2'])).T;Omega=np.cov(U,rowvar=False,bias=True);pv=float(v@Omega@v)
            assert abs(pv-r['variance'])<1e-8
            row.update(root_exposure=float(v[0]),H=float(v@v),variance_from_sources=pv)
        GROW.append(row)
    targets=[target_for(X,y,noise,dirs,W,P[:i],i,'base',beta) for i in range(len(W))]
    record_fit(seed,task,s,'growth',f'{topo}_beta{beta}',P,y,targets)


### I4. Teacher choice and scale extension

Preserve the first two analytic teachers. The third teacher has error amplitude
$0.3+\mathrm{gap}$ and direction $\gamma v_1+\sqrt{1-\gamma^2}v_3$,
where the fixed-seed direction $v_3$ is orthogonal to $v_1,v_2$.
The legacy CSV name `rho` denotes the paper's designed overlap $\gamma$.
For three teachers, the second and third error directions have mutual overlap
$\gamma^2$; their overlap with the first teacher is $\gamma$.

Assignments are selected on validation data and frozen before test evaluation.
The eight-student extension enumerates all $3^8=6561$ assignments. Relative gain is
computed per seed and then averaged. Balanced-combination diagnostics keep each
slot-teacher model's usage frequency fixed and decompose squared error into diagonal
and cross-error terms. Their comparator is the average concentrated assignment;
the optimized-choice comparator is the individually selected assignment.

In [ ]:
def teacher_functions(seed,task,gap,rho,T):
    X,theta,noise,dirs=DATA[seed];truth=truth_fn(X,theta,task);v1,v2=dirs;t=CFG['teacher_error']
    teachers=[truth+t*(X@v1),truth+(t+gap)*(X@(rho*v1+np.sqrt(1-rho*rho)*v2))]
    if T==3:
        # Preserve the first two teachers and use a fixed orthogonal third direction.
        r=np.random.default_rng(seed+400000);v3=r.normal(size=CFG['d']);v3-=dirs.T@(dirs@v3);v3/=np.linalg.norm(v3)
        teachers.append(truth+(t+gap)*(X@(rho*v1+np.sqrt(1-rho*rho)*v3)))
    return np.asarray(teachers),truth

def choice_bank(seed,task,gap,rho,S,T,s):
    case=dict(seed=int(seed),task=task,gap=float(gap),rho=float(rho),students=S,teachers=T,setting=s['id'],beta=CFG['beta'])
    path=RUN/'choice_predictions'/(digest(case)[:24]+'.npz')
    if path.is_file():
        z=read_npz(path);assert json.loads(str(z['case_json']))==case;return z,case
    teachers,truth=teacher_functions(seed,task,gap,rho,T);X,theta,noise,dirs=DATA[seed]
    if MODE=='full' and task=='linear' and S==4 and T==2 and s['id'] in ['paper_nn','ridge_005']:
        kind='mlp' if s['kind']=='mlp' else 'ridge';rel=f'choice_predictions/{seed}_{kind}_rho{rho}_gap{gap}.npz'
        if SOURCE.exists(rel):
            z=SOURCE.npz(rel);assert z['pred'].shape==(S,T,TOTAL)
            np.testing.assert_allclose(z['teachers'],teachers,rtol=1e-12,atol=1e-12);np.testing.assert_allclose(z['truth'],truth,rtol=1e-12,atol=1e-12)
            z.update(case_json=json.dumps(case),origin='original_choice');atomic_npz(path,**z);COUNTS['old_models']+=S*T;return z,case
    P=np.empty((S,T,TOTAL));ids=[]
    for i in range(S):
        local=local_target(X,truth,noise,i)
        for j in range(T):
            target=(1-CFG['beta'])*local+CFG['beta']*teachers[j]
            state,sig=fit_model(X,target,seed*100+i,s);P[i,j]=state['pred'];ids.append(sig)
    z=dict(pred=P,teachers=teachers,truth=truth,model_ids=np.asarray(ids),case_json=json.dumps(case),origin='new')
    atomic_npz(path,**z);return z,case

ASSIGNMENTS={}
def assignment_bank(S,T):
    if (S,T) not in ASSIGNMENTS:
        C=np.array(list(itertools.product(range(T),repeat=S)),int);Q=np.zeros((len(C),S*T));Q[np.arange(len(C))[:,None],np.arange(S)[None,:]*T+C]=1/S
        counts=np.stack([(C==j).sum(1) for j in range(T)],1)
        ASSIGNMENTS[S,T]=(C,Q,counts)
    return ASSIGNMENTS[S,T]

def evaluate_choice(seed,task,gap,rho,S,T,s,z,case,extension=False):
    P=z['pred'];truth=z['truth'];C,Q,counts=assignment_bank(S,T);ii=np.arange(S);E=P[:,:,VA]-truth[VA]
    private=np.mean(E**2,axis=2).argmin(1)
    if T==2 and S<=4:
        # Preserve the numerical enumeration order of the original four-student experiment.
        val=np.array([np.mean((P[ii,c][:,VA].mean(0)-truth[VA])**2) for c in C])
    else:
        V=E.reshape(S*T,-1);G=V@V.T/V.shape[1];val=np.einsum('ij,ij->i',Q@G,Q)
    group=C[int(np.argmin(val))]
    frozen=dict(case=case,selection='validation only',private=private.tolist(),group=group.tolist(),validation_group_losses=val.tolist())
    atomic_text(RUN/'frozen_choices'/(digest(case)[:24]+'.json'),json.dumps(frozen))
    # Evaluate test errors only after freezing assignments.
    E=P[:,:,TE]-truth[TE];flat=E.reshape(S*T,-1);G=flat@flat.T/flat.shape[1]
    losses=np.einsum('ij,ij->i',Q@G,Q)
    def loss(c):return float(np.mean((P[ii,c][:,TE].mean(0)-truth[TE])**2))
    lp,lg=loss(private),loss(group);own=(E**2).mean(2);margins=[]
    mm=dict(seed=seed,task=task,gap=gap,rho=rho,students=S,teachers=T,extension=extension,**meta(s))
    for i in range(S):
        alt=np.delete(own[i],private[i]);margin=float(np.min(alt)-own[i,private[i]]);margins.append(margin)
        MARGINS.append(dict(**mm,slot=i,chosen=int(private[i]),test_margin=margin))
    concentration=lambda c:float(sum(np.mean(c==j)**2 for j in range(T)))
    concentrated=len(set(private))==1;diversified=len(set(group))>1
    teacher_error=z['teachers'][:,TE]-truth[TE];tcorr=np.corrcoef(teacher_error)
    CHOICE.append(dict(**mm,private=json.dumps(private.tolist()),group=json.dumps(group.tolist()),private_mse=lp,group_mse=lg,gain=lp-lg,relative_gain=(lp-lg)/lp if lp>EPS else np.nan,private_H=concentration(private),group_H=concentration(group),private_concentrated=concentrated,group_diversified=diversified,all_test_margins_positive=min(margins)>0,joint_pattern=concentrated and diversified and min(margins)>0 and lp>lg,random_assignment_mse=float(losses.mean()),teacher_mse_all=json.dumps(np.mean(teacher_error**2,axis=1).tolist()),teacher_corr_all=json.dumps(tcorr.tolist()),teacher0_mse=float(np.mean(teacher_error[0]**2)),teacher1_mse=float(np.mean(teacher_error[1]**2)),measured_teacher_corr=float(tcorr[0,1])))
    same=np.max(counts,axis=1)==S;balanced=np.max(counts,axis=1)-np.min(counts,axis=1)<=1
    np.testing.assert_allclose(Q[same].mean(0),Q[balanced].mean(0),atol=1e-14)
    diagonal=(Q*Q)@np.diag(G);mu=flat.mean(1);bias=(Q@mu)**2;variance=losses-bias;cross=losses-diagonal
    ls,lm=float(losses[same].mean()),float(losses[balanced].mean());diag_gap=float(diagonal[same].mean()-diagonal[balanced].mean());cross_gain=float(cross[same].mean()-cross[balanced].mean())
    assert abs(diag_gap)<1e-10 and abs((ls-lm)-cross_gain)<1e-10
    BALANCED.append(dict(**mm,average_concentrated=ls,balanced=lm,gain=ls-lm,diagonal_difference=diag_gap,cross_error_gain=cross_gain,bias_squared_gain=float(bias[same].mean()-bias[balanced].mean()),variance_gain=float(variance[same].mean()-variance[balanced].mean()),n_balanced=int(balanced.sum())))
    X,theta,noise,dirs=DATA[seed]
    targets=[];pred=[]
    for i in range(S):
        local=local_target(X,truth,noise,i)
        for j in range(T):targets.append((1-CFG['beta'])*local+CFG['beta']*z['teachers'][j]);pred.append(P[i,j])
    record_fit(seed,task,s,'choice_extension' if extension else 'choice',f'rho{rho}_gap{gap}',np.asarray(pred),truth,targets)


### I5. Numerical checks and cache inventory
Check gradients, paired initialization, and available original-model predictions before running the experiments.

In [ ]:
# Check gradients and the zero-initialized linear skip connection.
rng=np.random.default_rng(42);xx=rng.normal(size=(11,8));yy=rng.normal(size=11)
for name in MAIN_IDS:
    s=SETTINGS[name];p=init_net(1234,s)
    if len(p)==5:p[4][:]=.3
    _,g=objective_grad(xx,yy,p,s['l2'])
    for k,idx in [(0,(1,2)),(1,(2,)),(2,(3,)),(3,(0,))]+([(4,(1,))] if len(p)==5 else []):
        old=p[k][idx];h=1e-6;p[k][idx]=old+h;up=objective_grad(xx,yy,p,s['l2'])[0]
        p[k][idx]=old-h;dn=objective_grad(xx,yy,p,s['l2'])[0];p[k][idx]=old
        assert abs((up-dn)/(2*h)-g[k][idx])<1e-6
p=init_net(1234,SETTINGS['plain_0']);q=init_net(1234,SETTINGS['residual_0'])
np.testing.assert_array_equal(forward(xx,p)[0],forward(xx,q)[0])
AUDIT=[]
if MODE=='full':
    seed=SEEDS[0];X,theta,noise,dirs=DATA[seed];s=SETTINGS['paper_nn']
    for task in TASKS:
        old=old_graph(seed,task,'star',8,.7,'base',s)
        if old:
            truth=truth_fn(X,theta,task);target=local_target(X,truth,noise,0,dirs,'base')
            fresh,_=fit_model(X,target,seed*100,s,allow_old=False)
            delta=fresh['pred']-old[0]['pred'][0];relative=np.sqrt(np.mean(delta**2))/max(np.sqrt(np.mean(old[0]['pred'][0]**2)),1e-12)
            AUDIT.append(dict(task=task,seed=seed,relative_rms_difference=float(relative),passed=bool(relative<1e-6)))
            assert relative<1e-6,'Original model reproduction exceeds tolerance; stopping'
csv('00_implementation_audit.csv',AUDIT)
# Inspect graph-cache availability before training.
plan_rows=[]
for sid in MAIN_IDS:
    s=SETTINGS[sid];found=0;total=0
    for seed in SEEDS:
        for task in TASKS:
            for topo in TOPOS:
                for variant in VARIANTS:
                    case=dict(seed=int(seed),task=task,topology=topo,M=PROP_M,beta=.7,variant=variant,setting=sid)
                    available=(RUN/'graph_predictions'/(digest(case)[:24]+'.npz')).is_file()
                    if not available:available=old_graph(seed,task,topo,PROP_M,.7,variant,s) is not None
                    total+=1;found+=int(available)
    plan_rows.append(dict(setting=sid,propagation_graphs=total,available_graphs=found,missing_graphs=total-found))
display(csv('00_propagation_cache_inventory.csv',plan_rows))
print('Gradient, initialization, and available original-model checks completed.')


### I6. Complete tables and checkpoint exports
Retain per-seed and per-node records for every configured condition. Summaries average within seeds before bootstrap resampling.

In [ ]:
RAW={'propagation_by_node.csv':PROP,'superposition.csv':SUPER,'conditional_risk.csv':RISK,'growth.csv':GROW,'choice.csv':CHOICE,'choice_margins.csv':MARGINS,'balanced_choice.csv':BALANCED,'fit_diagnostics.csv':FIT,'ridge_identity.csv':RIDGE,'roots_controls.csv':CONTROL,'calibrated_loadings.csv':LOADINGS}
def archive(final=False):
    path=RUN/(REPORT_KIND+('_tables_figures.zip' if final else '_partial.zip'));tmp=path.with_name(path.name+'.tmp')
    with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(REPORT.rglob('*')):
            if p.is_file() and not p.name.endswith('.tmp'):z.write(p,p.relative_to(REPORT))
    os.replace(tmp,path);return path

def save_progress(stage,seed=None):
    for name,rows in RAW.items():
        if rows:csv(name,rows)
    atomic_text(REPORT/'progress.json',json.dumps(dict(stage=stage,seed=seed,completed=DONE,counters=COUNTS,elapsed_seconds=time.time()-START),indent=2))
    atomic_text(REPORT/'source_files_sha256.json',json.dumps(SOURCE.accessed,indent=2));archive()

def summarize_all():
    global PROP_DF,GROW_DF,CHOICE_DF,BALANCED_DF,GROW_PAIRS,GROW_CHANGE
    for name,rows in RAW.items():csv(name,rows)
    PROP_DF=pd.DataFrame(PROP);GROW_DF=pd.DataFrame(GROW);CHOICE_DF=pd.DataFrame(CHOICE);BALANCED_DF=pd.DataFrame(BALANCED)
    ps=csv('01_propagation_summary.csv',summarize(PROP_DF,['task','setting','method'],['relative_rmse','mse','response_energy','skill']))
    csv('01_propagation_by_topology.csv',summarize(PROP_DF,['task','setting','method','topology','probe'],['relative_rmse','mse','response_energy']))
    csv('01_propagation_by_depth.csv',summarize(PROP_DF,['task','setting','method','topology','depth'],['relative_rmse','mse','response_energy']))
    pooled=[]
    for key,g in PROP_DF.groupby(['seed','task','setting','method']):
        good=np.isfinite(g.mse)&(g.response_energy>EPS);v=g[good];ratio=v.mse.sum()/v.response_energy.sum() if v.response_energy.sum()>EPS else np.nan
        pooled.append(dict(zip(['seed','task','setting','method'],key),pooled_relative_rmse=np.sqrt(ratio),pooled_skill=1-ratio,valid_rows=int(good.sum()),total_rows=len(g)))
    csv('01_pooled_by_seed.csv',pooled);csv('01_pooled_summary.csv',summarize(pd.DataFrame(pooled),['task','setting','method'],['pooled_relative_rmse','pooled_skill']))
    ss=csv('01_superposition_summary.csv',summarize(pd.DataFrame(SUPER),['task','setting','test'],['relative_rmse','mse','reference_energy']))
    csv('01_risk_summary.csv',summarize(pd.DataFrame(RISK).query("topology != 'roots'"),['task','setting','method'],['absolute_error','absolute_error_pct_baseline']))
    gs=csv('02_growth_summary.csv',summarize(GROW_DF,['task','setting','beta','topology','M'],['aggregate_mse','individual_mse','bias_squared','variance','offdiag_cov','effective_rank','standardized_aggregate_variance']))
    keys=['seed','task','setting','beta','M'];root=GROW_DF[GROW_DF.topology=='roots'][keys+['aggregate_mse','individual_mse']]
    paired=GROW_DF[GROW_DF.topology!='roots'].merge(root,on=keys,suffixes=('','_roots'),validate='many_to_one')
    paired['aggregate_difference']=paired.aggregate_mse-paired.aggregate_mse_roots;paired['individual_difference']=paired.individual_mse-paired.individual_mse_roots
    paired['aggregate_ratio']=paired.aggregate_mse/paired.aggregate_mse_roots;GROW_PAIRS=paired
    csv('02_growth_vs_roots.csv',paired);csv('02_growth_vs_roots_summary.csv',summarize(paired,['task','setting','beta','topology','M'],['aggregate_difference','individual_difference','aggregate_ratio']))
    keys=['seed','task','setting','beta','topology'];early=GROW_DF[GROW_DF.M==min(SIZES)][keys+['aggregate_mse','individual_mse']]
    end=GROW_DF[GROW_DF.M==max(SIZES)].merge(early,on=keys,suffixes=('_large','_small'),validate='one_to_one')
    end['aggregate_change']=end.aggregate_mse_large-end.aggregate_mse_small;end['individual_change']=end.individual_mse_large-end.individual_mse_small
    end['relative_aggregate_reduction']=1-end.aggregate_mse_large/end.aggregate_mse_small;GROW_CHANGE=end
    csv('02_growth_size_change.csv',end);csv('02_growth_size_change_summary.csv',summarize(end,['task','setting','beta','topology'],['aggregate_change','individual_change','relative_aggregate_reduction']))
    ck=['task','setting','extension','students','teachers','rho','gap']
    cs=csv('03_choice_summary.csv',summarize(CHOICE_DF,ck,['private_mse','group_mse','gain','relative_gain','joint_pattern','private_concentrated','group_diversified','all_test_margins_positive','random_assignment_mse']))
    csv('03_choice_margin_summary.csv',summarize(pd.DataFrame(MARGINS),ck,['test_margin']))
    bs=csv('03_balanced_summary.csv',summarize(BALANCED_DF,ck,['gain','cross_error_gain','bias_squared_gain','variance_gain','diagonal_difference']))
    csv('04_fit_summary.csv',summarize(pd.DataFrame(FIT),['task','setting','stage'],['train_mse','validation_mse','truth_test_mse']))
    # Audit the original 600-update neural propagation results.
    repro=ps[(ps.setting=='paper_nn')&(ps.method=='recursive')&(ps.metric=='relative_rmse')].copy()
    if MODE=='full':
        repro['paper_rounded']=repro.task.map({'linear':.420,'nonlinear':.714})
        repro['matches_rounding']=np.round(repro['mean'],3)==repro.paper_rounded
    csv('00_paper_reproduction.csv',repro)
    # Preserve the original experiment configuration.
    audits=[]
    for name,new,keys,metrics_ in [('growth.csv',GROW_DF,['seed','beta','topology','M'],['aggregate_mse','individual_mse']),('choice.csv',CHOICE_DF[~CHOICE_DF.extension],['seed','rho','gap'],['private_mse','group_mse','gain'])]:
        if MODE=='full' and SOURCE.exists(name):
            old=pd.read_csv(io.BytesIO(SOURCE.read(name)))
            for sid,kind,lam in [('paper_nn','mlp',0.),('ridge_005','ridge',.05)]+([('ridge_0','ridge',0.)] if name=='growth.csv' else []):
                a=new[(new.setting==sid)&(new.task=='linear')];b=old[old.learner==kind]
                if 'lam' in b:b=b[np.isclose(b.lam,lam)]
                merged=a.merge(b[keys+metrics_],on=keys,suffixes=('_new','_old'),validate='one_to_one')
                for metric in metrics_:
                    delta=np.max(np.abs(merged[metric+'_new']-merged[metric+'_old'])) if len(merged) else np.nan
                    audits.append(dict(table=name,setting=sid,metric=metric,matched_rows=len(merged),max_absolute_difference=delta,within_1e_7=bool(delta<1e-7)))
    csv('00_original_outcome_audit.csv',audits)
    rows=['# Mechanism supplementary experiments',
          f'Mode: {MODE}; independent seeds: {SEEDS}.',
          'Original data and target functions are retained. All configured settings are exported.',
          'Loadings and teacher assignments use validation data; evaluation uses the held-out test set.',
          'Means aggregate within seeds first. Bootstrap intervals are pointwise.',
          'The balanced-combination comparator is distinct from the individual-choice comparator.',
          'The CSV field rho denotes the designed direction overlap gamma.',
          'See 01_propagation_summary.csv, 02_growth_summary.csv, and 03_choice_summary.csv.']
    atomic_text(REPORT/'RESULTS.md','\n\n'.join(rows))
    return ps,ss,gs,cs,bs


### I7. Execute the mechanism experiments
Run propagation, growth, original-scale teacher choice, and the three-teacher extension. Matching graph and model caches are reused.

In [ ]:
# Save each seed and stage; reuse completed predictions on restart.
for seed in SEEDS:
    for task in TASKS:
        ids=MAIN_IDS+['paper_nn','ridge_005']+(['paper_nn_200'] if task=='linear' else [])
        for sid in ids:
            s=SETTINGS[sid]
            for topo in TOPOS:
                bank={v:graph(seed,task,topo,PROP_M,.7,v,s) for v in VARIANTS}
                evaluate_propagation(seed,task,topo,s,bank)
            log(f'Propagation {seed} {task} {sid}')
    DONE.append(f'propagation:{seed}');save_progress('propagation',seed)
for seed in SEEDS:
    for task in TASKS:
        ids=MAIN_IDS+['ridge_005','ridge_0']+(['paper_nn'] if task=='linear' else [])
        for sid in ids:
            s=SETTINGS[sid]
            for beta in BETAS:
                for topo in TOPOS:
                    z=graph(seed,task,topo,GROW_M,beta,'base',s);evaluate_growth(seed,task,topo,beta,s,z)
            log(f'Growth {seed} {task} {sid}')
    DONE.append(f'growth:{seed}');save_progress('growth',seed)
for seed in SEEDS:
    for task in TASKS:
        ids=MAIN_IDS+['ridge_005']+(['paper_nn'] if task=='linear' else [])
        for sid in ids:
            s=SETTINGS[sid]
            for rho in RHOS:
                for gap in GAPS:
                    z,case=choice_bank(seed,task,gap,rho,SLOTS,2,s);evaluate_choice(seed,task,gap,rho,SLOTS,2,s,z,case)
            log(f'Teacher choice {seed} {task} {sid}')
    DONE.append(f'choice:{seed}');save_progress('choice',seed)
if RUN_EXTENSION:
    for seed in SEEDS:
        for task in TASKS:
            for sid in MAIN_IDS+['ridge_005']:
                s=SETTINGS[sid]
                for rho in EXT_RHOS:
                    for gap in EXT_GAPS:
                        z,case=choice_bank(seed,task,gap,rho,EXT_SLOTS,3,s);evaluate_choice(seed,task,gap,rho,EXT_SLOTS,3,s,z,case,extension=True)
                log(f'Three-teacher extension {seed} {task} {sid}')
        DONE.append(f'extension:{seed}');save_progress('extension',seed)
PS,SS,GS,CS,BS=summarize_all()
print('All mechanism experiments and summaries completed.')


### I8. Focused inline results
Display the regularized propagation comparisons, topology and individual/group contrasts, and the ridge scale-extension example at gap=0.35 and gamma=0 (full mode). The full matched grid is saved. These conditions are fixed by the display specification; no result-sign filtering is applied.

In [ ]:
FIG=REPORT/'figures';FIG.mkdir(exist_ok=True)
plt.rcParams.update({'font.size':9,'axes.spines.top':False,'axes.spines.right':False})
def savefig(fig,name):
    fig.tight_layout();fig.savefig(FIG/(name+'.png'),dpi=180);fig.savefig(FIG/(name+'.pdf'));plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,task in zip(axes,TASKS):
    ids=['paper_nn']+MAIN_IDS;g=PS[(PS.task==task)&(PS.method=='recursive')&(PS.metric=='relative_rmse')].set_index('setting').reindex(ids)
    x=np.arange(len(ids));ax.bar(x,g['mean'],color=['#777777','#4488aa','#66aa99','#cc9966','#aa6677']);ax.vlines(x,g.ci_low,g.ci_high,color='black')
    ax.set_xticks(x,ids,rotation=25,ha='right');ax.set_title(task);ax.set_ylabel('Relative response RMSE')
savefig(fig,'01_propagation')
colors={'roots':'#0072B2','star':'#D55E00','chain':'#009E73','multi_parent':'#CC79A7'}
for beta in BETAS:
    fig,axes=plt.subplots(2,4,figsize=(15,7))
    for row,task in enumerate(TASKS):
        for col,sid in enumerate(MAIN_IDS):
            ax=axes[row,col]
            for topo in TOPOS:
                g=GS[(GS.task==task)&(GS.setting==sid)&(GS.beta==beta)&(GS.topology==topo)&(GS.metric=='aggregate_mse')].sort_values('M')
                ax.plot(g.M,g['mean'],'o-',label=topo,color=colors[topo]);ax.fill_between(g.M,g.ci_low,g.ci_high,alpha=.12,color=colors[topo])
            ax.set_title(task+' / '+sid);ax.set_xlabel('Models');ax.set_ylabel('Aggregate MSE')
    axes[0,0].legend(fontsize=7);savefig(fig,f'02_growth_beta_{beta}')
g=CS[(CS.metric=='gain')&(~CS.extension)&(CS.setting.isin(MAIN_IDS))];limit=max(float(g['mean'].abs().max()),1e-12)
fig,axes=plt.subplots(2,4,figsize=(15,6))
for row,task in enumerate(TASKS):
    for col,sid in enumerate(MAIN_IDS):
        ax=axes[row,col];q=g[(g.task==task)&(g.setting==sid)].pivot(index='rho',columns='gap',values='mean').reindex(index=RHOS,columns=GAPS)
        im=ax.imshow(q.to_numpy(),aspect='auto',cmap='RdBu',vmin=-limit,vmax=limit)
        ax.set_xticks(range(len(GAPS)),[str(x) for x in GAPS],rotation=30);ax.set_yticks(range(len(RHOS)),[str(x) for x in RHOS]);ax.set_title(task+' / '+sid);ax.set_xlabel('Teacher amplitude gap');ax.set_ylabel('Overlap')
        for i in range(len(RHOS)):
            for j in range(len(GAPS)):ax.text(j,i,f'{q.iloc[i,j]:.3f}',ha='center',va='center',fontsize=7)
savefig(fig,'03_teacher_choice_gain')
atomic_text(REPORT/'COMPLETE.json',json.dumps(dict(version=VERSION,mode=MODE,report_kind=REPORT_KIND,train_sha=TRAIN_SHA,seeds=SEEDS,main_settings=MAIN_IDS,extension=RUN_EXTENSION,completed=DONE,counters=COUNTS,elapsed_seconds=time.time()-START),indent=2))
ZIP=archive(final=True)
print('Mechanism tables and figures saved.')
MECHANISM_REPORT=REPORT
MECHANISM_RUN=RUN
if SOURCE.z:SOURCE.z.close()


In [ ]:
# Explicit display scope; full tables remain in the export.
DISPLAY_SETTINGS=['paper_nn','plain_l2','residual_l2']
print('Propagation: original and regularized neural settings, 600 updates')
prop_table=PS[(PS.method=='recursive')&(PS.metric=='relative_rmse')&PS.setting.isin(DISPLAY_SETTINGS)].copy()
pivot=prop_table.pivot(index='setting',columns='task',values='mean').reindex(DISPLAY_SETTINGS)
for task in TASKS:
    pivot[task+'_reduction_pct']=100*(1-pivot[task]/pivot.loc['paper_nn',task])
display(pivot)
csv('selected_propagation.csv',pivot.reset_index())

print('Population growth: beta=0.7, largest configured population')
g=GROW_DF[(GROW_DF.beta==.7)&(GROW_DF.M==max(SIZES))&GROW_DF.setting.isin(['plain_l2','residual_l2'])]
means=g.groupby(['task','setting','topology'])[['aggregate_mse','individual_mse']].mean()
display(means)
rows=[]
for (task,sid),q in g.groupby(['task','setting']):
    a=q.groupby('topology')[['aggregate_mse','individual_mse']].mean()
    roots=a.loc['roots'];star=a.loc['star']
    rows.append(dict(task=task,setting=sid,M=max(SIZES),beta=.7,
        individual_reduction_pct=100*(1-star.individual_mse/roots.individual_mse),
        aggregate_increase_pct=100*(star.aggregate_mse/roots.aggregate_mse-1)))
display(csv('selected_individual_population.csv',rows))

print('Scale extension: original ridge learner, linear task; gap=0.35 and gamma=0 in full mode')
q=CHOICE_DF[(CHOICE_DF.setting=='ridge_005')&(CHOICE_DF.task=='linear')&
            CHOICE_DF.gap.isin(EXT_GAPS)&CHOICE_DF.rho.isin(EXT_RHOS)]
scale=q.groupby(['gap','rho','teachers','students']).agg(
    mean_gain=('gain','mean'),mean_relative_gain=('relative_gain','mean'),
    joint_count=('joint_pattern','sum'),n_seeds=('seed','nunique')).reset_index()
scale['mean_relative_gain_pct']=100*scale.mean_relative_gain
csv('matched_scale_extension_grid.csv',scale)
selected_scale=scale[(scale.gap==.35)&(scale.rho==0.)] if MODE=='full' else scale
display(csv('selected_scale_extension.csv',selected_scale))

if SHOW_ALL_TABLES:
    for name in ['01_propagation_summary.csv','02_growth_summary.csv',
                 '03_choice_summary.csv','03_balanced_summary.csv','04_fit_summary.csv',
                 'matched_scale_extension_grid.csv']:
        print(name);display(pd.read_csv(REPORT/name))


## J. Combined export and reproducibility

The archive contains original tables/figures, optimizer tables/figures, and mechanism
tables/figures in separate directories, plus a machine-readable file inventory.
Large resumable model and graph caches remain in their existing locations.
All computed conditions, including the complete quality/overlap grids and diagnostics,
are exported. The inline tables are presentation selections, not a filter on the
saved observations. Existing cache compatibility identifiers are retained; they
identify the original numerical implementations rather than this edited notebook.


In [ ]:
EXPORT_ROOT=Path(os.environ.get('LINEAGE_EXPORT_OUTPUT',
    '/content/drive/MyDrive/lineage_anonymous_release' if IN_COLAB else './lineage_anonymous_release'))
EXPORT_ROOT.mkdir(parents=True,exist_ok=True)
combined=EXPORT_ROOT/'lineage_integrated_tables_figures.zip'
manifest=[]
with zipfile.ZipFile(combined,'w',zipfile.ZIP_DEFLATED) as bundle:
    for label,folder in [('original',PAPER_OUT),('optimizer',OPTIMIZER_REPORT),('mechanism',MECHANISM_REPORT)]:
        for p in sorted(folder.rglob('*')):
            rel=p.relative_to(folder)
            if not p.is_file() or any(part in {'data','model_cache','graph_predictions'} for part in rel.parts):continue
            if p.suffix.lower() not in {'.csv','.json','.md','.png','.pdf'}:continue
            name=label+'/'+rel.as_posix();blob=p.read_bytes()
            bundle.writestr(name,blob)
            manifest.append(dict(file=name,sha256=hashlib.sha256(blob).hexdigest(),bytes=len(blob)))
    bundle.writestr('file_inventory.json',json.dumps(manifest,indent=2))
print('Combined tables and figures:',combined)
print('Complete. Model caches remain available for resuming.')
if IN_COLAB and AUTO_DOWNLOAD:
    from google.colab import files
    files.download(str(combined))
